# EduAll notebook 2 of 2: explaining and comparing two grant allocations

**Course project:** EduAll Grant Call 2026, CHF 500,000 for 500 applicants.

**What this notebook does.** Your group allocated the budget; so did another group, whose method you do not know.
This notebook compares the two allocations and explains each of them, **your own included**. Knowing your own reasons
is not the same as knowing what your allocation does: a rule applied by hand to 500 applicants can drift, and a
weight that seemed decisive in discussion can turn out to matter little in the numbers. The notebook therefore
evaluates both allocations with the same code and the same standards.

**What it needs.** The two allocation files, and four data files that notebook 1 produced from the 500 forms (the
first cells download them for you): `EduAll_features.csv`, `EduAll_feature_dictionary.csv`, `EduAll_evidence.json` and `EduAll_forms.json`.

| Part | Question | Main methods |
|---|---|---|
| 1 | Do both allocations respect the rules of the call? | Audit of hard rules and conditions |
| 2 | How different are the two allocations, and for whom? | Overlap, rank agreement, Lorenz curves, group shares |
| 3 | How did each group treat the exceptions and grey zones? | Flagged groups, disputed cases with evidence from the forms |
| 4 | Which characteristics explain each allocation? | Surrogate models, SHAP, permutation importance, self-check of your own allocation |
| 5 | Which procedure could have produced each allocation? | Screens, selection rules, award formula, implied weights |
| 6 | How did each group interpret the soft criteria, and what did it trade off? | Readings of need, criteria profile |
| 7 | Do characteristics that no criterion justifies influence the awards? | Residual probe |
| 8 | How do the two allocations fare on the four graded dimensions? | Mission, distributive justice, representation, disparate impact |
| 9 | What does an LLM add? | Gemini as second reader, procedure reconstruction, devil's advocate, synthesis |
| 10 | What do we take into the report? | Summary, checklist, saved report |

**The notebook produces evidence; your group produces the judgement.** Every part ends with *How to read it*. The
report is graded on what you make of the evidence: which interpretation of the call each group adopted, whether it
is defensible, and what you concede.

**Before you start (Google Colab).**

1. Choose *File → Save a copy in Drive*. Colab opens your copy: work in it, so the executed notebook stays in your
   Drive for the hand-in.
2. For the LLM analyses of Part 9, store your Gemini API key: click the key icon (*Secrets*) in the left bar, add a
   secret named `GEMINI_API_KEY`, and switch on *Notebook access*.
3. Choose *Runtime → Run all* and accept the access to your Drive. The first cells create the folder
   `MyDrive/EduAll_part2`, download the four data files into it, and show two upload buttons: first choose your
   group's allocation file, then the file you received. Each file has two columns: student ID and amount in CHF.
4. The notebook then runs for two to three minutes. All tables, figures, LLM answers and an HTML report go to a new
   folder inside `MyDrive/EduAll_part2`.

A second run finds the data and your two files in the folder and goes straight to the analyses. To compare other
files, replace the two files in the folder, or write other file names in the configuration cell.


In [ ]:
# ============================ CONFIGURATION ============================
PROJECT_DIR      = "/content/drive/MyDrive/EduAll_part2"   # your Drive folder: the notebook creates it
own_allocation   = "our_allocation.csv"        # YOUR group's file (student ID, amount in CHF)
other_allocation = "received_allocation.csv" # the file you received from the other group
DATA_URL         = "https://raw.githubusercontent.com/AIforPhilanthropy/eduall/main/part2_compare_allocations"   # the course repository

# Leave these two empty: the comparison runs without a declared procedure.
OWN_DECLARED_PRIORITIES = {}
OWN_PROCEDURE = """"""

# Gemini (Part 9)
USE_GEMINI   = True                  # True: call Gemini with your key (Colab Secrets: GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.5-flash-lite"

# Thresholds used in Part 8 (each is explained where it is used)
SUFFICIENCY_GAP_CHF = 1000           # a documented gap is "closed" if less than this remains for the grant year
SIMILAR_NEIGHBOURS  = 5              # neighbours used to test whether similar applicants are treated alike
MIN_GROUP_SIZE      = 15             # groups smaller than this are not tested for disparate impact

In [ ]:
# Connect Google Drive, prepare the folder, and gather the six files the notebook needs.
import os, sys, importlib, subprocess, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:                                   # outside Colab (only used to test the notebook): the current folder
    PROJECT_DIR = os.environ.get("EDUALL_DIR", ".")
    DATA_URL = os.environ.get("EDUALL_DATA_URL", DATA_URL)
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working folder:", PROJECT_DIR)

# 1. The four data files, downloaded once from the course repository.
for name in ["EduAll_features.csv", "EduAll_feature_dictionary.csv", "EduAll_evidence.json", "EduAll_forms.json"]:
    if os.path.exists(name):
        print(f"   {name}: already in the folder")
    else:
        urllib.request.urlretrieve(f"{DATA_URL}/data/{name}", name)
        print(f"   {name}: downloaded")

# 2. The two allocation files. The example files come from the repository; for your own files, an upload button
#    appears for each file that the folder lacks.
for target, who in [(own_allocation, "YOUR group's allocation"), (other_allocation, "the allocation you RECEIVED")]:
    if os.path.exists(target):
        print(f"   {target}: already in the folder")
    elif target.endswith("_example.csv"):
        urllib.request.urlretrieve(f"{DATA_URL}/example/{os.path.basename(target)}", target)
        print(f"   {target}: downloaded (example)")
    elif IN_COLAB:
        from google.colab import files
        print(f"\nChoose the file with {who}. The notebook saves it in the folder as {target}.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Choose exactly one file, then run this cell again.")
        name, content = next(iter(uploaded.items()))
        with open(target, "wb") as fh:
            fh.write(content)
        if os.path.exists(name) and os.path.abspath(name) != os.path.abspath(target):
            os.remove(name)                    # Colab also keeps a copy under the original name
        print(f"   {target}: saved")
    else:
        raise FileNotFoundError(f"{target} is missing from {PROJECT_DIR}")

# 3. The stored example answers of Part 9, used when USE_GEMINI is False.
if not USE_GEMINI and not os.path.exists("llm_cache.json"):
    urllib.request.urlretrieve(f"{DATA_URL}/example/llm_cache.json", "llm_cache.json")

# 4. The two libraries that Colab may lack.
for pkg, mod in [("shap", "shap"), ("google-genai", "google.genai")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


In [ ]:
import base64, hashlib, io, itertools, json, re, textwrap, warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.stats import entropy, fisher_exact, spearmanr, ttest_ind
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import balanced_accuracy_score, r2_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 120)
OWN_COLOR, OTHER_COLOR, GREY, ACCENT = "#0b3c6f", "#6baed6", "#b0b7bf", "#d9822b"   # dark blue = own, light blue = other
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

# Where the results go: a folder named after the two allocation files.
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
OUTPUT_DIR = os.path.join(PROJECT_DIR, f"comparison_{stem(own_allocation)}_vs_{stem(other_allocation)}")
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "tables"), exist_ok=True)
REPORT = []                     # (kind, title, content) in order of appearance, for the HTML report of Part 10


def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")[:60]


def _fmt(v):
    """Readable numbers: thousands separators for large values, two decimals for shares, three digits for small p-values."""
    if isinstance(v, (bool, np.bool_)) or not isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    if pd.isna(v):
        return ""
    if float(v).is_integer() or abs(v) >= 100:
        return f"{v:,.0f}"
    return f"{v:.3g}" if abs(v) < 0.01 else f"{v:.2f}"


def show_table(df, caption=None, gradient=None):
    """Display a table, save it as CSV in OUTPUT_DIR/tables and keep it for the HTML report.
    `gradient`: columns to shade from light (low) to dark (high), to make large values easy to spot."""
    sty = df.style.format(_fmt)
    if gradient:
        sty = sty.background_gradient(cmap="Blues", subset=[c for c in gradient if c in df.columns])
    if caption:
        sty = sty.set_caption(caption).set_table_styles([{"selector": "caption",
                                                          "props": "font-weight:bold; font-size:110%; text-align:left"}])
        df.to_csv(os.path.join(OUTPUT_DIR, "tables", slug(caption) + ".csv"))
        REPORT.append(("table", caption, sty.to_html()))
    try:
        from IPython.display import display
        display(sty)
    except ImportError:
        print(caption or "", df, sep="\n")


def show_figure(name, caption=""):
    """Save the current figure in OUTPUT_DIR/figures, keep it for the HTML report, and display it."""
    fig = plt.gcf()
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "figures", slug(name) + ".png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    REPORT.append(("figure", name, (base64.b64encode(buf.getvalue()).decode(), caption)))
    plt.show()
    plt.close(fig)


print("Results will be saved in", OUTPUT_DIR)

### Loading what notebook 1 produced

Notebook 1 read the 500 forms and saved four files, which come in the student pack. We load them here and rebuild the objects that the analyses
use: `CSV` (the 43 original columns), `F` (the variables derived from the forms), `EVIDENCE` (the sentences behind
each derived variable) and `FORMS` (the complete forms, for the LLM). The dictionary of derived variables tells you
what every column means; it is printed so that you can consult it while reading.

In [ ]:
needed = ["EduAll_features.csv", "EduAll_feature_dictionary.csv", "EduAll_evidence.json", "EduAll_forms.json"]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Not found in {PROJECT_DIR}: {missing}. Run the setup cell at the top again: it downloads them.")

FEATURES = pd.read_csv("EduAll_features.csv").set_index("student_id")
DICTIONARY = pd.read_csv("EduAll_feature_dictionary.csv").set_index("variable")
EVIDENCE = json.load(open("EduAll_evidence.json", encoding="utf-8"))
_forms = json.load(open("EduAll_forms.json", encoding="utf-8"))
FORMS = {sid: v["form"] for sid, v in _forms.items()}
P = pd.DataFrame({sid: {k: v for k, v in d.items() if k != "form"} for sid, d in _forms.items()}).T

DERIVED = [c for c in DICTIONARY.index if c in FEATURES.columns]
CSV, F = FEATURES.drop(columns=DERIVED), FEATURES[DERIVED].copy()
for c in F.columns:                                   # empty text values are read back as missing: restore them
    if F[c].dtype == object:
        F[c] = F[c].fillna("")

BUDGET = 500_000                                                        # Grant Call, Executive Summary
LEVEL_MAX = {"Bachelor's": 8000, "Master's": 6500, "Doctoral": 5000}    # Grant Call, Available Funding (CHF per year)
RISK_FACTORS = ["risk_first_generation", "risk_refugee_asylum", "risk_disability", "risk_housing_insecurity",
                "risk_family_emergency", "risk_language_barrier", "risk_parent_student", "risk_undocumented_status"]
RESILIENCE_FACTORS = ["resilience_volunteer_experience", "resilience_work_experience", "resilience_leadership_role",
                      "resilience_community_engagement", "resilience_overcame_adversity"]
print(f"{len(CSV)} applicants; {CSV.shape[1]} original columns; {F.shape[1]} derived variables; {len(FORMS)} forms.")
show_table(DICTIONARY[["group", "meaning"]], "Variables derived from the forms in notebook 1")

---
# Part 1. Loading the two allocations and auditing them

Numbers in brackets such as (2.6) refer to the sections of notebook 1 where a variable was derived and justified.

## 1.1 Loading and checking the files

An allocation file must contain one row per student, with the student ID in the first column and the amount in CHF
in the second. The loader below accepts any column names, removes thousands separators, treats missing students as
"not funded" and reports every problem it finds. **Read its messages**: a typo in an ID silently drops a student.

In [ ]:
def load_allocation(path, label):
    raw = pd.read_csv(path)
    if raw.shape[1] < 2:
        raise ValueError(f"{path}: expected two columns (student ID, amount), found {raw.shape[1]}")
    ids = raw.iloc[:, 0].astype(str).str.strip()
    amounts = pd.to_numeric(raw.iloc[:, 1].astype(str).str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")
    problems = {
        "IDs not in the dataset": sorted(set(ids) - set(CSV.index))[:5],
        "duplicate IDs": sorted(ids[ids.duplicated()].unique())[:5],
        "amounts that are not numbers": int(amounts.isna().sum()),
        "negative amounts": int((amounts < 0).sum()),
    }
    s = pd.Series(amounts.values, index=ids).groupby(level=0).sum()
    s = s.reindex(CSV.index).fillna(0).clip(lower=0).rename(label)
    print(f"{label}: {path} -> {int((s > 0).sum())} students funded, CHF {s.sum():,.0f} in total")
    for name, value in problems.items():
        if value:
            print(f"   WARNING, {name}: {value}")
    return s


OWN = load_allocation(own_allocation, "own")
OTHER = load_allocation(other_allocation, "other")
PAIR = pd.DataFrame({"own": OWN, "other": OTHER})

## 1.2 Audit: hard rules, conditions of the call, and choices that need a justification

We start with the audit because every later comparison assumes that both allocations solve the *same* problem. An
allocation that overspends has more money to distribute and would look better on every later indicator, for the
wrong reason. The audit sorts the checks into three levels:

| Level | Meaning | Checks |
|---|---|---|
| **Hard rule** | No interpretation possible. A breach is an error. | Budget of CHF 500,000; maximum per study level |
| **Condition of the call** | The call states a rule but allows exceptions or conditions. Funding needs a stated reason. | Award above the documented ceiling (2.5); Permit C rule applies (2.6); doctoral funding gap not documented (2.9); GPA below 2.5 without explanation (2.11) |
| **Conditional award** | Allowed by the call, but only under a condition the group should mention. | Pending permit (2.8) |

In [ ]:
CEILING, LMAX = F["ceiling_text"], CSV["study_level"].map(LEVEL_MAX)
CHECKS = {
    ("hard rule", "awards above the level maximum"): lambda a: a > LMAX,
    ("condition", "award above the documented ceiling (+CHF 100)"): lambda a: a > CEILING + 100,
    ("condition", "Permit C, family resources, rule applies"): lambda a: (a > 0) & (F["permitC_family_resources"] == "rule applies"),
    ("condition", "doctoral funding gap not documented"): lambda a: (a > 0) & (F["doctoral_funding_documented"] == 0),
    ("condition", "GPA below 2.5, not explained"): lambda a: (a > 0) & (F["low_gpa_status"] == "not explained"),
    ("condition", "Permit B, study permit conflict"): lambda a: (a > 0) & (F["study_permit_conflict"] == 1),
    ("conditional", "pending permit (award conditional on the permit)"): lambda a: (a > 0) & (F["permit_pending"] == 1),
}


def audit(a):
    out = {("hard rule", "total (CHF)"): a.sum(), ("hard rule", "over the CHF 500,000 budget"): a.sum() > BUDGET}
    for key, test in CHECKS.items():
        mask = test(a)
        amount = (a - CEILING).clip(lower=0)[mask].sum() if "ceiling" in key[1] else \
                 (a - LMAX).clip(lower=0)[mask].sum() if "maximum" in key[1] else a[mask].sum()
        out[(key[0], key[1] + ": students")] = int(mask.sum())
        out[(key[0], key[1] + ": CHF")] = round(amount)
    return pd.Series(out)


AUDIT = pd.DataFrame({"own": audit(OWN), "other": audit(OTHER)})
show_table(AUDIT, "Audit of the two allocations")

**How to read it.** Any entry on a *hard rule* row other than zero (or `False`) is an error to report to the other
group first, before any discussion of values. Entries on *condition* rows are not errors, but each one needs a
reason that the group can state; ask for it. *Conditional* awards are fine if the group made them conditional.

---
# Part 2. Exploratory data analysis: how different are the two allocations?

Before asking *why* the allocations differ, we measure *how much* they differ, and for whom. Three numbers summarise
the disagreement, from coarse to fine:

* **Overlap of recipients (Jaccard index):** students funded by both, divided by students funded by at least one.
  1 means the same recipients; 0 means none in common.
* **Rank agreement (Spearman correlation):** do students who receive more from one group also receive more from
  the other?
* **Money moved:** the CHF one would have to take from some students and give to others to turn one allocation
  into the other (half the sum of absolute differences).

The **Gini coefficient** describes concentration: 0 means everyone receives the same; values close to 1 mean that
few students receive almost everything.

In [ ]:
def gini(a):
    x = np.sort(np.asarray(a, dtype=float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())) if x.sum() > 0 else 0.0


def profile(a):
    f = a[a > 0]
    return pd.Series({"students funded": len(f), "total CHF": a.sum(), "mean award": f.mean(), "median award": f.median(),
                      "largest award": a.max(), "Gini (all 500)": gini(a), "Gini (recipients)": gini(f)})


def distance(a, b):
    fa, fb = a > 0, b > 0
    return pd.Series({"funded by both": int((fa & fb).sum()), "funded only by own": int((fa & ~fb).sum()),
                      "funded only by other": int((~fa & fb).sum()), "funded by neither": int((~fa & ~fb).sum()),
                      "Jaccard overlap": (fa & fb).sum() / max(1, (fa | fb).sum()),
                      "Spearman rank agreement": spearmanr(a, b)[0],
                      "CHF moved": 0.5 * (a - b).abs().sum(), "share of budget moved": 0.5 * (a - b).abs().sum() / BUDGET})


show_table(pd.DataFrame({"own": profile(OWN), "other": profile(OTHER)}).round(2), "Shape of each allocation")
show_table(distance(OWN, OTHER).round(3).to_frame("value"), "Disagreement between the two allocations")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
both = (OWN > 0) & (OTHER > 0)
ax[0].scatter(OWN[~both], OTHER[~both], s=9, color=GREY, label="funded by one or none")
ax[0].scatter(OWN[both], OTHER[both], s=9, color=OWN_COLOR, label="funded by both")
lim = max(OWN.max(), OTHER.max()) * 1.05
ax[0].plot([0, lim], [0, lim], color="black", lw=0.6, ls="--")
ax[0].set(xlabel="own allocation (CHF)", ylabel="other allocation (CHF)", title="Same student, two decisions")
ax[0].legend(frameon=False, fontsize=7)
bins = np.linspace(1, lim, 25)
ax[1].hist(OWN[OWN > 0], bins=bins, color=OWN_COLOR, alpha=0.9, label="own")
ax[1].hist(OTHER[OTHER > 0], bins=bins, color=OTHER_COLOR, alpha=0.8, label="other")
ax[1].set(xlabel="award size (CHF)", ylabel="number of awards", title="Breadth versus depth")
ax[1].legend(frameon=False)
for s, c in [(OWN, OWN_COLOR), (OTHER, OTHER_COLOR)]:
    x = np.sort(s.values)
    ax[2].plot(np.linspace(0, 1, len(x)), np.cumsum(x) / x.sum(), color=c, label=f"{s.name} (Gini {gini(s):.2f})")
ax[2].plot([0, 1], [0, 1], color="black", lw=0.6, ls="--")
ax[2].set(xlabel="applicants, smallest to largest award", ylabel="cumulative share of budget", title="Concentration (Lorenz curve)")
ax[2].legend(frameon=False)
show_figure("2 Same student two decisions, award sizes, Lorenz curves")

**How to read the panels.** Left: each dot is one applicant; dots on the dashed diagonal receive the same from both
groups, dots on an axis are funded by one group only. Middle: many small awards mean the group chose breadth; few
large awards mean depth. Right: the further a curve bends away from the diagonal, the more the money is
concentrated.

Next, *for whom* do the allocations differ? We split each budget by applicant group. A large difference in a
group's share is a first hint at a different rationale.

In [ ]:
def by_group(col, values=None):
    g = (values if values is not None else CSV[col]).rename(col)
    t = pd.DataFrame({"applicants": g.value_counts(),
                      "own: share of budget": OWN.groupby(g).sum() / OWN.sum(),
                      "other: share of budget": OTHER.groupby(g).sum() / OTHER.sum(),
                      "own: share funded": (OWN > 0).groupby(g).mean(),
                      "other: share funded": (OTHER > 0).groupby(g).mean()})
    return t.sort_values("applicants", ascending=False)


for col in ["study_level", "gni_tier", "residency_permit", "gender"]:
    show_table(by_group(col), f"Shares by {col}", gradient=["own: share funded", "other: share funded"])

---
# Part 3. Exceptions and grey zones: where the reading of the forms matters

Notebook 1 identified groups of applicants for whom the call leaves a decision to the committee: an exception may apply,
a condition may be missing, or the form contains information that changes the need. This part shows what each
allocation decided for each group. Here, more than anywhere else, the two groups' interpretations of the soft
criteria become visible.

## 3.1 Decisions per group of flagged applicants

In [ ]:
FLAGS = {
    "Permit C, rule applies (2.6)": F["permitC_family_resources"] == "rule applies",
    "Permit C, exception candidate (2.6)": F["permitC_family_resources"] == "exception candidate",
    "pending permit (2.8)": F["permit_pending"] == 1,
    "doctoral gap not documented (2.9)": F["doctoral_funding_documented"] == 0,
    "GPA < 2.5, explained and improving (2.11)": F["low_gpa_status"] == "explained and improving",
    "GPA < 2.5, explained only (2.11)": F["low_gpa_status"] == "explained, no improvement shown",
    "GPA < 2.5, not explained (2.11)": F["low_gpa_status"] == "not explained",
    "family support at risk (2.4)": F["family_support_at_risk"] == 1,
    "scholarship ends before grant year (2.3)": F["scholarship_status"] == "ends before grant year",
    "savings set aside for dependants or care (2.2)": F["savings_earmark"].str.contains("dependants|medical"),
    "states no current gap (2.13)": F["states_no_current_gap"] == 1,
    "finishes in Jan 2028, 5 months to fund (2.1)": F["months_grant_year"] <= 5,
    "part-time Permit B, basis not stated (2.7)": F["permit_basis_unclear_part_time"] == 1,
    "risk boxes explained only generically (2.14)": F["risk_ticks_generic"] > 0,
}


def flag_table():
    rows = []
    for name, mask in FLAGS.items():
        row = {"flag": name, "applicants": int(mask.sum())}
        for lab, a in [("own", OWN), ("other", OTHER)]:
            row[f"{lab}: funded"] = int((a[mask] > 0).sum())
            row[f"{lab}: CHF"] = round(a[mask].sum())
        row["funded by one group only"] = int(((OWN[mask] > 0) != (OTHER[mask] > 0)).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("flag")


FLAG_TABLE = flag_table()
show_table(FLAG_TABLE, "How each allocation treats the flagged groups")

**How to read it.** Compare the two *funded* columns row by row. A row where one group funds many and the other few
reveals a different interpretation: for example, funding pending permits conditionally versus excluding them, or
accepting a low GPA only when the applicant shows improvement. The last column counts the applicants on whom the two
groups disagree. Those are the cases to discuss.

## 3.2 The disputed cases, with the evidence from the forms

For the flagged applicants funded by one group only, the table below shows the two amounts, the flags and the
sentence from the form behind each flag. Read a few: would you defend your group's decision on this case?

In [ ]:
FLAG_EVIDENCE = {   # which evidence list explains which flag
    "Permit C, exception candidate (2.6)": ["family support at risk", "family support withdrawn"],
    "pending permit (2.8)": ["grant as proof of means"],
    "GPA < 2.5, explained and improving (2.11)": ["low GPA improving"],
    "GPA < 2.5, explained only (2.11)": ["low GPA explained"],
    "family support at risk (2.4)": ["family support at risk"],
    "scholarship ends before grant year (2.3)": ["scholarship ends"],
    "savings set aside for dependants or care (2.2)": ["savings for dependants", "savings for family medical costs"],
    "states no current gap (2.13)": ["states no current gap"],
    "doctoral gap not documented (2.9)": [],
}


def why(sid, flags):
    for flag in flags:
        for key in FLAG_EVIDENCE.get(flag, []):
            if sid in EVIDENCE.get(key, {}):
                return EVIDENCE[key][sid][0]
    return textwrap.shorten(P.at[sid, "financial_statement"], 160, placeholder=" ...")


rows = []
for sid in CSV.index:
    flags = [name for name, mask in FLAGS.items() if mask[sid]]
    if flags and (OWN[sid] > 0) != (OTHER[sid] > 0):
        rows.append({"student": sid, "own": OWN[sid], "other": OTHER[sid], "ceiling": CEILING[sid],
                     "flags": "; ".join(flags), "evidence": textwrap.shorten(why(sid, flags), 170, placeholder=" ...")})
DISPUTED = pd.DataFrame(rows).set_index("student") if rows else pd.DataFrame()
if len(DISPUTED):
    DISPUTED = DISPUTED.reindex((DISPUTED["own"] - DISPUTED["other"]).abs().sort_values(ascending=False).index)
    print(f"{len(DISPUTED)} flagged applicants are funded by one group only. The 15 largest differences:")
    show_table(DISPUTED.head(15), "Disputed flagged applicants (15 largest differences)")

---
# Part 4. Explainable AI: which characteristics explain each allocation?

Parts 2 and 3 showed *that* the allocations differ and *for whom*. Explainable AI asks *which applicant
characteristics* produce the difference. The logic has three links.

1. **Surrogate model.** We do not know the rule the other group used. We therefore train a model that predicts the
   group's amounts from the applicant data. If the model predicts well, it has learned an imitation of the group's
   rule, called a *surrogate*.
2. **Fidelity.** A surrogate explains an allocation only as far as it reproduces it. We measure this with R² on
   applicants the model has not seen (5-fold cross-validation): 1 means perfect imitation, 0 means none. Low
   fidelity is itself a finding: the allocation depends on something the features do not capture.
3. **Explanation.** A shallow *decision tree* gives readable if-then rules. A *gradient-boosting* model explained with
   **SHAP values** gives, for every applicant, the contribution in CHF of each characteristic to the predicted amount.
   SHAP values come from cooperative game theory (Shapley values): they split a prediction fairly among the features
   that produced it.

## 4.1 Which features, and why

A surrogate can only reveal reliance on a characteristic that it is given. We give it three kinds:

* **criteria from the CSV**: the financial, academic, residence and circumstance variables that the call names;
* **criteria from the forms (notebook 1)**: months in the grant year, savings left and set aside, scholarship ending,
  family support at risk, the Permit C status, the doctoral condition, the GPA explanation, the candour signals and
  the number of specific risk boxes. If an allocation relied on reading the forms, these features will matter;
* **characteristics that are not criteria**: age, gender and statement length. The call's equal-opportunity policy
  excludes gender; no criterion mentions age or length. We include them so that the surrogate can *show* reliance
  on them if it exists; Part 7 then tests it directly.

In [ ]:
CSV_CRITERIA = ["financial_need_chf", "family_income_chf", "family_household_income_chf", "personal_income_chf",
                "expected_personal_income_2027_chf", "monthly_expenses_chf", "savings_chf", "external_scholarship_amount_chf",
                "gni_tier", "gpa", "courses_completed", "publications", "dependent_children"] + RISK_FACTORS + RESILIENCE_FACTORS
FORM_CRITERIA = ["months_grant_year", "savings_left_sep2027", "family_support_at_risk", "states_no_current_gap",
                 "accepts_lower_priority", "risk_ticks_specific", "ceiling_text"]
NON_CRITERIA = ["age", "statement_words"]
X = pd.concat([
    CSV[CSV_CRITERIA + ["age"]],
    pd.get_dummies(CSV[["study_level", "study_mode", "residency_permit"]], dtype=int),
    F[FORM_CRITERIA + ["statement_words"]],
    pd.DataFrame({"scholarship_ends_before_2027": (F["scholarship_status"] == "ends before grant year").astype(int),
                  "savings_set_aside": F["savings_earmark"].str.contains("dependants|medical").astype(int),
                  "permitC_rule_applies": (F["permitC_family_resources"] == "rule applies").astype(int),
                  "permitC_exception": (F["permitC_family_resources"] == "exception candidate").astype(int),
                  "doctoral_gap_undocumented": (F["doctoral_funding_documented"] == 0).astype(int),
                  "low_gpa_explained": F["low_gpa_status"].str.startswith("explained").astype(int),
                  "gender_female": (CSV["gender"] == "Female").astype(int),
                  "gender_male": (CSV["gender"] == "Male").astype(int)}),
], axis=1)
X.columns = [c.replace("_chf", "") for c in X.columns]
FEATURE_KIND = {c: ("form" if c in [x.replace("_chf", "") for x in FORM_CRITERIA] + ["scholarship_ends_before_2027", "savings_set_aside",
                    "permitC_rule_applies", "permitC_exception", "doctoral_gap_undocumented", "low_gpa_explained"]
                    else "not a criterion" if c in ["age", "statement_words", "gender_female", "gender_male"] else "CSV")
                for c in X.columns}
print(X.shape[1], "features:", pd.Series(FEATURE_KIND).value_counts().to_dict())


def fit_surrogate(y):
    tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=0).fit(X, y)
    gbm = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=0)
    cv_pred = pd.Series(cross_val_predict(gbm, X, y, cv=5), index=X.index)   # each applicant predicted by a model that has not seen it
    gbm.fit(X, y)
    sv = pd.DataFrame(shap.TreeExplainer(gbm).shap_values(X), index=X.index, columns=X.columns)
    return {"tree": tree, "tree_r2": tree.score(X, y), "fidelity": r2_score(y, cv_pred), "shap": sv, "gbm": gbm, "cv_pred": cv_pred, "y": y}


def drivers(sur, k=8):
    imp = sur["shap"].abs().mean().sort_values(ascending=False).head(k)
    sign = [np.sign(np.corrcoef(X[f], sur["shap"][f])[0, 1]) if X[f].std() > 0 and sur["shap"][f].std() > 0 else 0 for f in imp.index]
    return pd.DataFrame({"mean |SHAP| (CHF)": imp.round(0), "kind": [FEATURE_KIND[f] for f in imp.index],
                         "higher value means": ["more money" if v > 0 else "less money" for v in sign]})


S_OWN, S_OTHER = fit_surrogate(OWN), fit_surrogate(OTHER)
print(f"Surrogate fidelity (cross-validated R²): own {S_OWN['fidelity']:.2f}, other {S_OTHER['fidelity']:.2f}")

## 4.2 Readable rules: the surrogate decision trees

Each line is a question about the applicant; `value` is the average amount for applicants who reach that leaf. The
first split is the single characteristic that best separates high from low awards.

In [ ]:
for sur, lab in [(S_OWN, "own"), (S_OTHER, "other")]:
    print(f"--- {lab}: the tree reproduces {sur['tree_r2']:.0%} of the variation in amounts ---")
    txt = export_text(sur["tree"], feature_names=list(X.columns), decimals=2)
    print(txt)
    REPORT.append(("text", f"4.2 Surrogate tree, {lab} allocation (R² {sur['tree_r2']:.2f})", txt))

## 4.3 What drives each allocation (SHAP), side by side

The bars show by how many CHF, on average, each characteristic moves an applicant's predicted amount. The column
`kind` tells you whether the feature comes from the CSV, from the forms, or is not a criterion at all.

In [ ]:
def plot_drivers(sa, sb, k=12):
    ia, ib = sa["shap"].abs().mean(), sb["shap"].abs().mean()
    top = (ia / ia.sum() + ib / ib.sum()).sort_values(ascending=False).head(k).index[::-1]
    y = np.arange(len(top))
    fig, ax = plt.subplots(figsize=(8, 4.4))
    ax.barh(y + 0.2, ia[top], height=0.4, color=OWN_COLOR, label=f"own (fidelity {sa['fidelity']:.2f})")
    ax.barh(y - 0.2, ib[top], height=0.4, color=OTHER_COLOR, label=f"other (fidelity {sb['fidelity']:.2f})")
    ax.set_yticks(y); ax.set_yticklabels([f"{t}  [{FEATURE_KIND[t]}]" for t in top])
    ax.set(xlabel="mean |SHAP value| (CHF)", title="What drives each allocation?")
    ax.legend(frameon=False); show_figure("4.3 What drives each allocation (SHAP)")


plot_drivers(S_OWN, S_OTHER)
show_table(drivers(S_OWN), "Drivers of the own allocation")
show_table(drivers(S_OTHER), "Drivers of the other allocation")

## 4.4 Contrastive explanation: what explains the *difference*?

A sharper question than "what drives each allocation" is: *why does this student receive more from one group than
from the other?* We answer it by training a surrogate on the difference `own − other`. A positive SHAP value now means
"this characteristic pushes the student towards the own allocation".

In [ ]:
S_DIFF = fit_surrogate(OWN - OTHER)
print(f"Fidelity of the contrastive surrogate: {S_DIFF['fidelity']:.2f}")
shap.summary_plot(S_DIFF["shap"].values, X, max_display=12, show=False, plot_size=(8, 4.5))
plt.title("Right: favoured by own allocation; left: favoured by other allocation", fontsize=9)
show_figure("4.4 Contrastive SHAP own minus other")

**How to read the plot.** Each dot is one applicant. Colour shows the value of the characteristic (red high, blue
low). Dots to the right receive more from your group because of this characteristic; dots to the left receive more
from the other group. Example: if red dots of `ceiling_text` lie to the right, your group pays more to applicants
with a large documented shortfall.

## 4.5 Local explanations: the largest disagreements

Global patterns can hide individual cases. For the three largest disagreements in each direction, the table lists
the characteristics that contribute most to the disagreement, and the first sentence of the Financial Need
Statement, so that you can connect the numbers to the person.

In [ ]:
diff = (OWN - OTHER).sort_values()
rows = []
for sid in list(diff.index[-3:][::-1]) + list(diff.index[:3]):
    top = S_DIFF["shap"].loc[sid].abs().sort_values(ascending=False).head(3).index
    rows.append({"student": sid, "own": OWN[sid], "other": OTHER[sid],
                 "main contributions (CHF)": "; ".join(f"{f}={X.at[sid, f]:g} ({S_DIFF['shap'].at[sid, f]:+,.0f})" for f in top),
                 "form": textwrap.shorten(P.at[sid, "financial_statement"], 150, placeholder=" ...")})
LOCAL = pd.DataFrame(rows).set_index("student")
show_table(LOCAL, "Largest disagreements and what explains them")

## 4.6 Self-check: does your allocation follow your own rule?

**Why.** Your group knows the rule it intended. The surrogate of your allocation (4.1) learned the rule your
allocation *actually* follows. Where the two diverge for an individual applicant, there are two possible reasons:
a deliberate exception (the form contains something that justifies departing from the rule), or an inconsistency
(similar applicants were treated differently by accident, for example because different members read different
forms). Both deserve a sentence in the report; only the first is defensible.

**How.** For each applicant, the surrogate predicts the amount from the other 400 applicants of its fold, without
having seen this applicant (5-fold cross-validation, as in 4.1). The **deviation** is the actual amount minus this
prediction. A large positive deviation means: "your allocation pays this applicant much more than it pays applicants
who look like it". We list the largest deviations in both directions, with the flags of 3.1 and the sentence of the
form behind them. We run the same check on the other allocation, because its deviations are good questions for the
other group.

In [ ]:
def rule_deviations(a, sur, k=6):
    dev = (a - sur["cv_pred"]).sort_values()
    rows = []
    for sid in list(dev.index[-k:][::-1]) + list(dev.index[:k]):
        flags = [name for name, mask in FLAGS.items() if mask[sid]]
        rows.append({"student": sid, "direction": "paid more than the rule predicts" if dev[sid] > 0 else "paid less than the rule predicts",
                     "award": a[sid], "rule predicts": round(sur["cv_pred"][sid]), "deviation": round(dev[sid]),
                     "documented ceiling": CEILING[sid], "flags": "; ".join(flags) or "none",
                     "evidence from the form": textwrap.shorten(why(sid, flags), 150, placeholder=" ...")})
    return pd.DataFrame(rows).set_index("student")


DEV_OWN, DEV_OTHER = rule_deviations(OWN, S_OWN), rule_deviations(OTHER, S_OTHER)
show_table(DEV_OWN, "Self-check: where your allocation departs most from its own rule")
show_table(DEV_OTHER, "Where the other allocation departs most from its own rule")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for axis, a, sur, c, lab in [(ax[0], OWN, S_OWN, OWN_COLOR, "own"), (ax[1], OTHER, S_OTHER, OTHER_COLOR, "other")]:
    dev = a - sur["cv_pred"]
    big = dev.abs() >= dev.abs().quantile(0.97)
    axis.scatter(sur["cv_pred"], a, s=8, color=c, alpha=0.6)
    axis.scatter(sur["cv_pred"][big], a[big], s=22, facecolors="none", edgecolors=ACCENT, label="largest 3% of deviations")
    lim = max(a.max(), sur["cv_pred"].max()) * 1.05
    axis.plot([0, lim], [0, lim], color="black", lw=0.6, ls="--")
    axis.set(xlabel="amount predicted by the allocation's own rule (CHF)", ylabel="actual award (CHF)",
             title=f"{lab}: fidelity R² = {sur['fidelity']:.2f}")
    axis.legend(frameon=False, fontsize=7)
show_figure("4.6 Self-check: actual award versus the allocation's own rule")

**How to read it.** Points on the dashed diagonal are applicants whom the allocation treats exactly as its rule
predicts. Circled points are the largest departures. For every applicant in *your* table, your group should be able
to say "we did this on purpose, because ..." and point to the form. If you cannot, you have found an inconsistency;
saying so in the report is a strength, not a weakness. A surrogate with low fidelity produces many large deviations:
then the rule itself is not captured by the features, and the deviations say more about the features than about
your consistency.

## 4.7 Two ways to measure importance, and do they agree?

**Why.** SHAP (4.3) is one way to measure how much an allocation relies on a characteristic. A conclusion such as
"the other group relies on GPA" should not depend on the method. **Permutation importance** (Breiman, 2001; Fisher,
Rudin and Dominici, 2019) measures reliance differently: it shuffles one feature across applicants, which breaks its
link with the amounts, and records how much the surrogate's accuracy drops. A large drop means the surrogate needs
that feature.

**A caution that matters here.** Several features measure almost the same thing: the CSV's `financial_need`, the
three readings of need and the documented ceiling are strongly correlated. Correlated features *share* credit, in
both methods, and the split between them is partly arbitrary (Aas, Jullum and Løland, 2021; Hooker, Mentch and Zhou,
2021). Two allocations that both rely on need can therefore show different need features at the top. Section 4.8
groups features by criterion, which removes most of this problem.

**What we compare.** (a) The two methods within each allocation: if they rank the features similarly (Spearman ρ
close to 1), the importance ranking is robust. (b) The two allocations with each method: a low ρ means the two
groups relied on different characteristics.

In [ ]:
def permutation_reliance(sur):
    r = permutation_importance(sur["gbm"], X, sur["y"], n_repeats=5, random_state=0, scoring="r2")
    return pd.Series(r.importances_mean, index=X.columns).clip(lower=0)


PERM_OWN, PERM_OTHER = permutation_reliance(S_OWN), permutation_reliance(S_OTHER)
SHAP_OWN, SHAP_OTHER = S_OWN["shap"].abs().mean(), S_OTHER["shap"].abs().mean()
IMPORTANCE = pd.DataFrame({"own: SHAP share": SHAP_OWN / SHAP_OWN.sum(), "other: SHAP share": SHAP_OTHER / SHAP_OTHER.sum(),
                           "own: permutation share": PERM_OWN / max(PERM_OWN.sum(), 1e-9),
                           "other: permutation share": PERM_OTHER / max(PERM_OTHER.sum(), 1e-9)})
IMPORTANCE["kind"] = [FEATURE_KIND[c] for c in IMPORTANCE.index]
IMPORTANCE = IMPORTANCE.sort_values("own: SHAP share", ascending=False)
AGREEMENT = pd.DataFrame({
    "own vs other (SHAP)": [spearmanr(SHAP_OWN, SHAP_OTHER)[0]],
    "own vs other (permutation)": [spearmanr(PERM_OWN, PERM_OTHER)[0]],
    "SHAP vs permutation (own)": [spearmanr(SHAP_OWN, PERM_OWN)[0]],
    "SHAP vs permutation (other)": [spearmanr(SHAP_OTHER, PERM_OTHER)[0]]}, index=["Spearman ρ"]).T
show_table(AGREEMENT, "Agreement between importance rankings")
show_table(IMPORTANCE.head(15), "Importance of the 15 features that matter most for the own allocation",
           gradient=["own: SHAP share", "other: SHAP share", "own: permutation share", "other: permutation share"])

fig, ax = plt.subplots(figsize=(6.4, 5.6))
kind_color = {"CSV": OWN_COLOR, "form": ACCENT, "not a criterion": "#c0392b"}
for kind, col in kind_color.items():
    m = IMPORTANCE["kind"] == kind
    ax.scatter(IMPORTANCE.loc[m, "own: SHAP share"], IMPORTANCE.loc[m, "other: SHAP share"], s=28, color=col, label=kind)
top = (IMPORTANCE["own: SHAP share"] + IMPORTANCE["other: SHAP share"]).sort_values(ascending=False).head(8).index
for f in top:
    ax.annotate(f, (IMPORTANCE.at[f, "own: SHAP share"], IMPORTANCE.at[f, "other: SHAP share"]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")
lim = IMPORTANCE[["own: SHAP share", "other: SHAP share"]].values.max() * 1.1
ax.plot([0, lim], [0, lim], color="black", lw=0.6, ls="--")
ax.set(xlabel="share of SHAP importance, own allocation", ylabel="share of SHAP importance, other allocation",
       title="Importance map: above the diagonal = matters more for the other group", xlim=(0, lim), ylim=(0, lim))
ax.legend(frameon=False, fontsize=8)
show_figure("4.7 Importance map own versus other")

**How to read it.** In the map, each dot is one feature. A feature far above the diagonal matters for the other
allocation and not for yours; far below, the reverse. Orange dots come from the forms: if they lie near zero for one
allocation, that group probably decided from the CSV alone. Red dots are not criteria. If the two methods disagree
strongly about one allocation (ρ below about 0.5), rely on the grouped view of 4.8 rather than on single features.

## 4.8 Importance by criterion of the call, compared with the call's weights and with your declared priorities

**Why.** The call states weights: financial need 40%, academic motivation and potential 25%, resilience 20%,
diversity 10%, values 5%. A group's report usually states priorities too. Neither is the same as the weight a
criterion *carries* in the allocation. We therefore group the features by criterion and measure each group's share
of the explanation.

**How.** SHAP values add up: for each applicant, the SHAP values of all features sum to the predicted amount minus
the average. The sum of the SHAP values of the features of one criterion is therefore that criterion's
contribution for that applicant. We average its absolute value over applicants and divide by the total. The mapping
of features to criteria below is **our interpretation**; edit `FAMILY` if your group reads the call differently (for
example, if you count the risk boxes under resilience). Values (5%) has no measurable feature and is left out; the
call's weights are rescaled to the four measurable criteria (40, 25, 20, 10 → 42%, 26%, 21%, 11%). Two families have
no weight in the call: *eligibility and award rules* (permits, study level and mode), which act as gates and caps
rather than as weights, and *not a criterion*, whose share should be close to zero.

In [ ]:
NEED_FEATURES = ["financial_need", "family_income", "family_household_income", "personal_income", "expected_personal_income_2027",
                 "monthly_expenses", "savings", "external_scholarship_amount", "dependent_children", "months_grant_year",
                 "savings_left_sep2027", "family_support_at_risk", "states_no_current_gap", "accepts_lower_priority", "ceiling_text",
                 "scholarship_ends_before_2027", "savings_set_aside"]
FAMILY = {c: ("financial need" if c in NEED_FEATURES
              else "academic" if c in ["gpa", "courses_completed", "publications", "low_gpa_explained"]
              else "resilience" if c in RESILIENCE_FACTORS
              else "diversity & circumstances" if c in RISK_FACTORS + ["gni_tier", "risk_ticks_specific"]
              else "not a criterion" if FEATURE_KIND[c] == "not a criterion"
              else "eligibility & award rules") for c in X.columns}
CRITERIA4 = ["financial need", "academic", "resilience", "diversity & circumstances"]
CALL_WEIGHTS = pd.Series({"financial need": 40, "academic": 25, "resilience": 20, "diversity & circumstances": 10}) / 95


def family_shares(sur):
    grouped = sur["shap"].T.groupby(pd.Series(FAMILY)).sum().T          # applicant x family contributions
    imp = grouped.abs().mean()
    return imp / imp.sum()


FAM_OWN, FAM_OTHER = family_shares(S_OWN), family_shares(S_OTHER)
FAMILIES = pd.DataFrame({"own: share of all": FAM_OWN, "other: share of all": FAM_OTHER}).fillna(0)
for lab, fam in [("own", FAM_OWN), ("other", FAM_OTHER)]:
    FAMILIES[f"{lab}: share among the 4 criteria"] = (fam.reindex(CRITERIA4) / fam.reindex(CRITERIA4).sum()).reindex(FAMILIES.index)
FAMILIES["call weight (rescaled)"] = CALL_WEIGHTS.reindex(FAMILIES.index)
DECLARED = pd.Series(OWN_DECLARED_PRIORITIES, dtype=float)
if len(DECLARED):
    DECLARED = DECLARED.reindex(CRITERIA4).fillna(0) / DECLARED.sum()
    FAMILIES["own: declared priority"] = DECLARED.reindex(FAMILIES.index)
FAMILIES = FAMILIES.reindex(CRITERIA4 + ["eligibility & award rules", "not a criterion"])


def tvd(p, q):
    """Total variation distance between two weight vectors: the share of weight that would have to move. 0 = identical."""
    p, q = p.reindex(CRITERIA4).fillna(0), q.reindex(CRITERIA4).fillna(0)
    return 0.5 * (p / p.sum() - q / q.sum()).abs().sum()


DIST_TO_CALL = {lab: tvd(FAMILIES[f"{lab}: share among the 4 criteria"], CALL_WEIGHTS) for lab in ["own", "other"]}
show_table(FAMILIES, "Weight carried by each criterion (grouped SHAP)",
           gradient=["own: share among the 4 criteria", "other: share among the 4 criteria"])
print("Distance from the call's weights (total variation; 0 = same weights, 1 = no overlap): "
      + ", ".join(f"{k} {v:.2f}" for k, v in DIST_TO_CALL.items())
      + (f"; own measured vs own declared {tvd(FAMILIES['own: share among the 4 criteria'], DECLARED):.2f}" if len(DECLARED) else ""))

cols = [c for c in ["call weight (rescaled)", "own: declared priority", "own: share among the 4 criteria",
                    "other: share among the 4 criteria"] if c in FAMILIES]
colors = {"call weight (rescaled)": GREY, "own: declared priority": "white", "own: share among the 4 criteria": OWN_COLOR,
          "other: share among the 4 criteria": OTHER_COLOR}
fig, ax = plt.subplots(figsize=(9, 3.8))
x, wdt = np.arange(len(CRITERIA4)), 0.8 / len(cols)
for i, c in enumerate(cols):
    ax.bar(x + (i - (len(cols) - 1) / 2) * wdt, FAMILIES.loc[CRITERIA4, c], width=wdt, color=colors[c],
           edgecolor=OWN_COLOR if c == "own: declared priority" else "none", hatch="//" if c == "own: declared priority" else None,
           label=c.replace(": share among the 4 criteria", ": measured"))
ax.set_xticks(x); ax.set_xticklabels(CRITERIA4)
ax.set(ylabel="share of weight", title="Call weights, declared priorities and the weight each criterion actually carries")
ax.legend(frameon=False, fontsize=8, ncol=2)
show_figure("4.8 Criterion weights: call, declared, measured")

**How to read it.** Compare the bars within each criterion. The grey bar is what the call asks for, the hatched bar
what your group intended, the dark bar what your allocation does, the light bar what the other allocation does. A
gap between your hatched and dark bars is the most useful finding of the self-check: either your procedure did not
implement your intention (for example, a criterion you discussed never entered the ranking), or the intention was
implemented through another variable (for example, resilience counted through the risk boxes). A large share for
*eligibility & award rules* means that much of the variation in amounts comes from caps and gates, which is normal
when awards are capped by study level. Keep in mind that a criterion can be important in the call and still carry
little weight in an allocation if it barely varies among the applicants who compete for the money.

## 4.9 How the amount responds to the most important characteristics

**Why.** Importance says *how much* a characteristic matters, not *how*. The shape of the response reveals the way a
group used it: a step means a threshold (a screen or a tier), a straight line means proportional treatment, a curve
that flattens means a cap.

**How.** For the three characteristics most important across both allocations, each dot is one applicant: its value
on the horizontal axis, its SHAP value (the CHF this characteristic adds to or removes from its predicted amount) on
the vertical axis.

In [ ]:
TOP3 = (IMPORTANCE["own: SHAP share"] + IMPORTANCE["other: SHAP share"]).sort_values(ascending=False).head(3).index.tolist()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for axis, f in zip(ax, TOP3):
    jitter = np.random.default_rng(0).normal(0, X[f].std() * 0.02 + 1e-9, len(X)) if X[f].nunique() <= 5 else 0
    axis.scatter(X[f] + jitter, S_OWN["shap"][f], s=7, color=OWN_COLOR, alpha=0.6, label="own")
    axis.scatter(X[f] + jitter, S_OTHER["shap"][f], s=7, color=OTHER_COLOR, alpha=0.6, label="other")
    axis.axhline(0, color="black", lw=0.5)
    axis.set(xlabel=f"{f}  [{FAMILY[f]}]", ylabel="SHAP value (CHF)", title=f)
    axis.legend(frameon=False, fontsize=7)
show_figure("4.9 Response of the amount to the three most important characteristics")

**How to read it.** Look at the shape of each cloud. Example: if the dark cloud for `ceiling_text` rises steadily
while the light cloud jumps once and then stays flat, your group pays in proportion to the documented gap, whereas
the other group funds everyone above a threshold with a similar amount. Name the shape in your report and connect it
to a sentence of the call.

---
# Part 5. Retro-engineering: which procedure could have produced each allocation?

Part 4 explained each allocation with a flexible model. A flexible model imitates well but does not *read* like a
procedure. A grant committee, a script or an AI agent usually proceeds in steps: it **screens** some applicants out,
**selects** among the rest, and **sets an amount** for each recipient. This part reconstructs these three steps from
the allocation file alone, then **replays** the reconstructed procedure to test whether it reproduces the
allocation. The method is known in judgement research as *policy capturing*: inferring the policy of a decision
maker from its decisions (Karren and Barringer, 2002). We apply it to your allocation and to the other one in the same
way; for yours, you can check the reconstruction against what you actually did.

## 5.1 Screens: which groups of applicants were excluded, penalised or favoured?

**How.** For each group that the call or the forms single out, we compare the share funded inside the group with the
share funded among all other applicants. The verdict follows simple thresholds: *excluded* if the inside rate is
below one tenth of the outside rate; *penalised* below one half; *favoured* above one and a half; otherwise *no
visible effect*. The p-value (Fisher's exact test) states how often a difference this large would arise by chance;
small groups (for example 16 pending permits) need large differences to be convincing.

In [ ]:
SCREEN_CANDIDATES = {
    "no documented shortfall (ceiling = 0)": F["ceiling_text"] <= 0,
    "states no current gap": F["states_no_current_gap"] == 1,
    "pending permit": F["permit_pending"] == 1,
    "Permit C, rule applies": F["permitC_family_resources"] == "rule applies",
    "Permit C, exception candidate": F["permitC_family_resources"] == "exception candidate",
    "GPA below 2.5 (any)": F["gpa_below_2_5"] == 1,
    "GPA below 2.5, not explained": F["low_gpa_status"] == "not explained",
    "doctoral, funding gap not documented": F["doctoral_funding_documented"] == 0,
    "study permit conflict": F["study_permit_conflict"] == 1,
    "part-time study (CSV)": CSV["study_mode"] == "Part-time",
    "has an external scholarship": CSV["has_external_scholarship"] == 1,
    "finishes by Jan 2028 (5 months or fewer)": F["months_grant_year"] <= 5,
    "family support at risk": F["family_support_at_risk"] == 1,
    "first-generation student": CSV["risk_first_generation"] == 1,
    "GNI tier 1 country": CSV["gni_tier"] == 1,
    "refugee or asylum background": CSV["risk_refugee_asylum"] == 1,
    "dependent children": CSV["dependent_children"] > 0,
}


def screens(a):
    f, rows = a > 0, {}
    for name, m in SCREEN_CANDIDATES.items():
        if m.sum() == 0 or (~m).sum() == 0:
            continue
        inside, outside = f[m].mean(), f[~m].mean()
        ratio = inside / outside if outside > 0 else np.nan
        verdict = ("excluded" if ratio < 0.1 else "penalised" if ratio < 0.5 else "favoured" if ratio > 1.5 else "no visible effect")
        p = fisher_exact([[int(f[m].sum()), int((~f[m]).sum())], [int(f[~m].sum()), int((~f[~m]).sum())]])[1]
        rows[name] = {"applicants": int(m.sum()), "funded inside": inside, "funded outside": outside, "ratio": ratio,
                      "p-value": p, "verdict": verdict if p < 0.05 or verdict == "no visible effect" else verdict + " (uncertain)"}
    return pd.DataFrame(rows).T


SCR_OWN, SCR_OTHER = screens(OWN), screens(OTHER)
SCREENS_TABLE = pd.concat({"own": SCR_OWN[["funded inside", "funded outside", "verdict"]],
                           "other": SCR_OTHER[["funded inside", "funded outside", "verdict"]]}, axis=1)
SCREENS_TABLE.insert(0, "applicants", SCR_OWN["applicants"])
show_table(SCREENS_TABLE, "Inferred screens: share funded inside and outside each group")

**How to read it.** A row where one allocation says *excluded* and the other *no visible effect* is a different
reading of the call, usually the clearest single difference between two groups. Check each verdict against your own
procedure: if your table says *penalised* for a group you never discussed, the penalty came in through another
variable (for example, pending applicants often have low documented need). *Uncertain* means the difference could be
chance.

## 5.2 Selection: who gets anything at all?

**How.** We train a shallow classification tree that predicts *funded or not* from the features of 4.1. Each leaf of
the tree is a rule (a combination of conditions) with the share of its applicants who are funded. Rules with a share
near 0% or 100% describe the selection step crisply. We report the balanced accuracy of the tree on applicants it
has not seen (0.5 = no better than guessing, 1 = perfect) and the AUC of a flexible classifier, which states how
predictable the selection is at all.

In [ ]:
def selection_rules(a, depth=3):
    y = (a > 0).astype(int)
    tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=15, class_weight="balanced",
                                  min_impurity_decrease=0.005, random_state=0)
    bal_acc = balanced_accuracy_score(y, cross_val_predict(tree, X, y, cv=5))
    auc = roc_auc_score(y, cross_val_predict(GradientBoostingClassifier(random_state=0), X, y, cv=5, method="predict_proba")[:, 1])
    tree.fit(X, y)
    t, rows = tree.tree_, []

    def walk(node, conds):
        if t.children_left[node] == -1:
            idx = tree.apply(X) == node
            rows.append({"rule": " AND ".join(conds) or "all applicants", "applicants": int(idx.sum()),
                         "share funded": y[idx].mean(), "share of all recipients": y[idx].sum() / max(1, y.sum())})
            return
        f, thr = X.columns[t.feature[node]], t.threshold[node]
        binary = set(np.unique(X[f])) <= {0, 1}
        walk(t.children_left[node], conds + [f"{f} = 0" if binary else f"{f} ≤ {thr:,.2f}"])
        walk(t.children_right[node], conds + [f"{f} = 1" if binary else f"{f} > {thr:,.2f}"])

    walk(0, [])
    rules = pd.DataFrame(rows).sort_values("share funded", ascending=False).reset_index(drop=True)
    return {"rules": rules, "balanced accuracy": bal_acc, "AUC": auc}


SEL_OWN, SEL_OTHER = selection_rules(OWN), selection_rules(OTHER)
for sel, lab in [(SEL_OWN, "own"), (SEL_OTHER, "other")]:
    print(f"{lab}: balanced accuracy of the 3-level tree {sel['balanced accuracy']:.2f}; AUC of a flexible classifier {sel['AUC']:.2f}")
    show_table(sel["rules"], f"Selection rules, {lab} allocation", gradient=["share funded"])

**How to read it.** Read the rules from the top: the first rows describe who is almost always funded, the last rows
who is almost never funded. A high AUC with a low balanced accuracy means that the selection is predictable but not
by three simple conditions: the group combined many characteristics, as a score does. A low AUC (below about 0.7)
means that the selection depends on something the features do not capture, such as a reading of the texts that
notebook 1 did not encode.

## 5.3 Amount: what formula sets the award of each recipient?

**How.** Among recipients, we look for the fingerprint of the common award formulas:

| Formula | Fingerprint |
|---|---|
| flat amount | most recipients receive the same sum |
| a few tiers | six or fewer distinct amounts |
| fill each documented gap | most awards equal the documented ceiling (within CHF 100) |
| proportional to need | awards follow the ceiling closely (ρ ≥ 0.6) and the share of the gap paid varies little |
| need-scaled by a score | awards follow the ceiling (ρ ≥ 0.4), but the share of the gap paid varies |
| not tied to documented need | none of the above |

In [ ]:
def award_formula(a):
    f = a > 0
    amt, ceil, lmax = a[f], CEILING[f], F.loc[f, "level_max_prorated"]
    ratio = (amt / ceil.where(ceil > 0)).dropna()
    fp = {"recipients": int(f.sum()), "distinct amounts": int(amt.round(-1).nunique()),
          "share at the most common amount": amt.round(-1).value_counts(normalize=True).iloc[0],
          "share at the documented ceiling (±100)": ((amt - ceil).abs() <= 100).mean(),
          "share at the level maximum (±100)": ((amt - lmax).abs() <= 100).mean(),
          "share above the ceiling": (amt > ceil + 100).mean(),
          "median share of the gap paid": ratio.median() if len(ratio) else np.nan,
          "spread of the share paid (IQR)": ratio.quantile(0.75) - ratio.quantile(0.25) if len(ratio) else np.nan,
          "ρ award vs ceiling": spearmanr(amt, ceil)[0] if f.sum() > 2 else np.nan}
    fp["formula"] = ("flat amount" if fp["share at the most common amount"] >= 0.6
                     else "a few tiers" if fp["distinct amounts"] <= 6
                     else "fill each documented gap" if fp["share at the documented ceiling (±100)"] >= 0.6
                     else "proportional to need" if fp["ρ award vs ceiling"] >= 0.6 and fp["spread of the share paid (IQR)"] <= 0.3
                     else "need-scaled by a score" if fp["ρ award vs ceiling"] >= 0.4
                     else "not tied to documented need")
    return pd.Series(fp)


FORMULA = pd.DataFrame({"own": award_formula(OWN), "other": award_formula(OTHER)})
show_table(FORMULA, "Fingerprint of the award formula (recipients only)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for axis, a, c, lab in [(ax[0], OWN, OWN_COLOR, "own"), (ax[1], OTHER, OTHER_COLOR, "other")]:
    f = a > 0
    axis.scatter(CEILING[~f], a[~f], s=5, color=GREY, alpha=0.4, label="not funded")
    axis.scatter(CEILING[f], a[f], s=9, color=c, label="funded")
    m = max(CEILING.max(), a.max())
    axis.plot([0, m], [0, m], color="black", lw=0.6, ls="--", label="award = documented ceiling")
    axis.set(xlabel="documented ceiling, ceiling_text (CHF)", ylabel="award (CHF)", title=f"{lab}: {FORMULA.at['formula', lab]}")
    axis.legend(frameon=False, fontsize=7)
show_figure("5.3 Award versus documented ceiling")

**How to read it.** Dots on the dashed line receive exactly their documented ceiling; dots below it receive part of
their gap; dots above it receive more than the forms document. Grey dots on the horizontal axis are applicants with a
documented gap who receive nothing. A horizontal band of dots is a flat amount; a fan below the diagonal is a score
that scales the gap.

## 5.4 Implied weights: how much does each criterion of the call move an applicant up the ranking?

**Why.** 4.8 measured weights through a flexible model. Here we use the classic, transparent alternative: a linear
model of the ranking. Dawes (1979) showed that simple linear models reproduce expert judgements surprisingly well,
which is why policy capturing relies on them. A linear model also gives the weights the same form as the call
(percentages that add up to 100).

**How.** We score every applicant between 0 and 1 on each of the four measurable criteria: *financial need* (rank of
the documented shortfall, notebook 1, 2.5), *academic* (rank of the GPA), *resilience* (share of the five resilience
boxes ticked) and *diversity & circumstances* (average of: low-income country, first-generation, number of
substantiated risk boxes, notebook 1, 2.14). We regress the rank of each applicant's award on the four standardised
scores. The size of each coefficient, as a share of the total, is the criterion's **implied weight**; a negative
coefficient means the criterion lowers the award. We repeat the regression on 200 bootstrap samples to obtain a 90%
interval for each weight.

In [ ]:
SCORES = pd.DataFrame({
    "financial need": F["shortfall_text"].rank(pct=True),
    "academic": CSV["gpa"].rank(pct=True),
    "resilience": CSV[RESILIENCE_FACTORS].sum(axis=1) / 5,
    "diversity & circumstances": ((CSV["gni_tier"] == 1).astype(float) + CSV["risk_first_generation"]
                                  + F["risk_ticks_specific"].clip(upper=4) / 4) / 3})
Z = (SCORES - SCORES.mean()) / SCORES.std()


def implied_weights(a, n_boot=200):
    y = a.rank(pct=True)
    def fit(idx):
        b = LinearRegression().fit(Z.loc[idx], y[idx]).coef_
        return b / np.abs(b).sum()
    lin = LinearRegression().fit(Z, y)
    rng = np.random.default_rng(0)
    boot = np.array([fit(rng.choice(Z.index, len(Z))) for _ in range(n_boot)])
    w = lin.coef_ / np.abs(lin.coef_).sum()
    return pd.DataFrame({"implied weight": w, "90% low": np.quantile(boot, 0.05, axis=0),
                         "90% high": np.quantile(boot, 0.95, axis=0)}, index=Z.columns), lin.score(Z, y)


(IW_OWN, R2_IW_OWN), (IW_OTHER, R2_IW_OTHER) = implied_weights(OWN), implied_weights(OTHER)
IW = pd.DataFrame({"call weight (rescaled)": CALL_WEIGHTS, "own: implied": IW_OWN["implied weight"],
                   "own: 90% interval": [f"{l:.2f} to {h:.2f}" for l, h in zip(IW_OWN["90% low"], IW_OWN["90% high"])],
                   "other: implied": IW_OTHER["implied weight"],
                   "other: 90% interval": [f"{l:.2f} to {h:.2f}" for l, h in zip(IW_OTHER["90% low"], IW_OTHER["90% high"])]})
if len(DECLARED):
    IW.insert(1, "own: declared", DECLARED)
show_table(IW, "Implied weights of the four criteria (linear policy capturing)")
print(f"Share of the variation in award ranks explained by the four scores: own R² = {R2_IW_OWN:.2f}, other R² = {R2_IW_OTHER:.2f}")

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(CRITERIA4))
ax.bar(x - 0.27, CALL_WEIGHTS[CRITERIA4], 0.25, color=GREY, label="call weight (rescaled)")
for off, iw, c, lab in [(0, IW_OWN, OWN_COLOR, "own: implied"), (0.27, IW_OTHER, OTHER_COLOR, "other: implied")]:
    v = iw.loc[CRITERIA4]
    ax.bar(x + off, v["implied weight"], 0.25, color=c, label=lab,
           yerr=[v["implied weight"] - v["90% low"], v["90% high"] - v["implied weight"]], capsize=3, ecolor="black")
if len(DECLARED):
    ax.scatter(x, DECLARED[CRITERIA4], marker="D", color=ACCENT, zorder=3, label="own: declared")
ax.axhline(0, color="black", lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(CRITERIA4)
ax.set(ylabel="weight (share of total)", title=f"Implied weights (R² own {R2_IW_OWN:.2f}, other {R2_IW_OTHER:.2f})")
ax.legend(frameon=False, fontsize=8, ncol=2)
show_figure("5.4 Implied weights of the criteria")

**How to read it.** The error bars show how precisely the data determine each weight; two weights whose intervals
overlap are not reliably different. Read R² first: if the four scores explain little of the ranking (say, below
0.3), the allocation follows something other than a weighted sum of the criteria (for example, screens followed by
equal amounts), and the weights describe only a small part of it. When 4.8 and 5.4 agree, the conclusion is robust;
when they disagree, say which one you trust and why (4.8 sees non-linear rules; 5.4 is easier to read).

## 5.5 The reconstructed procedure, and a replay test

**How.** We write the three steps found above as a procedure and run it: (1) exclude the groups found *excluded* in
5.1; (2) rank the remaining applicants by the linear score of 5.4 and fund as many as the allocation funds; (3) set
each amount with the formula of 5.3, then rescale to the allocation's total. The **replay** shows how close this
simple procedure comes to the real allocation. If it comes close, the reconstruction is a credible account of the
rule. If not, the allocation contains something the three steps miss, and that is where to ask questions.

In [ ]:
def replay(a, scr):
    f = a > 0
    excluded = pd.Series(False, index=a.index)
    for name in scr.index[scr["verdict"] == "excluded"]:
        excluded |= SCREEN_CANDIDATES[name]
    score = pd.Series(LinearRegression().fit(Z, a.rank(pct=True)).predict(Z), index=a.index).where(~excluded)   # the 5.4 score
    chosen = score.dropna().nlargest(int(f.sum())).index
    kind, amt = award_formula(a)["formula"], pd.Series(0.0, index=a.index)
    if kind in ("flat amount", "a few tiers"):
        amt[chosen] = a[f].median()
    elif kind == "fill each documented gap":
        amt[chosen] = CEILING[chosen]
    elif kind in ("proportional to need", "need-scaled by a score"):
        amt[chosen] = (a[f] / CEILING[f].where(CEILING[f] > 0)).median() * CEILING[chosen]
    else:
        amt[chosen] = a[f].median()
    amt = (amt * a.sum() / max(amt.sum(), 1)).clip(upper=CSV["study_level"].map(LEVEL_MAX))
    lin = LinearRegression().fit(Z, a.rank(pct=True))
    weights = pd.Series(lin.coef_ / np.abs(lin.coef_).sum(), index=Z.columns)
    steps = {"1 screens (excluded groups)": ", ".join(scr.index[scr["verdict"] == "excluded"]) or "none",
             "2 selection": f"fund the top {int(f.sum())} by the score " + " ".join(f"{w:+.2f}·{c}" for c, w in weights.items()),
             "3 amount": kind}
    fa, fb = f, amt > 0
    fit = {"recipients in common": int((fa & fb).sum()), "Jaccard overlap of recipients": (fa & fb).sum() / max(1, (fa | fb).sum()),
           "Spearman ρ of amounts": spearmanr(a, amt)[0], "share of budget the replay misplaces": 0.5 * (a - amt).abs().sum() / a.sum()}
    return amt, steps, fit


REPLAY = {}
for a, scr, lab in [(OWN, SCR_OWN, "own"), (OTHER, SCR_OTHER, "other")]:
    REPLAY[lab] = replay(a, scr)
show_table(pd.DataFrame({lab: {**REPLAY[lab][1], **REPLAY[lab][2]} for lab in REPLAY}), "Reconstructed procedure and replay test")

**How to read it.** The top rows are the reconstructed procedure; the bottom rows measure the replay. As a rule of
thumb, a Jaccard overlap above about 0.7 and a misplaced share below about 0.3 mean that the three steps capture the
allocation well. For **your** allocation, compare the reconstruction with what you did: a screen you applied that is
missing here, or a screen here that you never applied, is worth a sentence in the report. For the **other**
allocation, the reconstruction is a hypothesis about its method that you can put to the other group. Part 9 asks an
LLM to write both reconstructions in words and to compare yours with the procedure you declared.

---
# Part 6. Interpretation of the soft criteria and trade-offs

xAI speaks in columns. The call speaks in criteria: financial need, academic potential, resilience, diversity. A
criterion becomes a number only after someone decides what it means (interpretation) and how to compute it
(implementation). This part makes those decisions visible.

## 6.1 Which reading of financial need does each allocation follow?

Notebook 1 (2.5) computed three readings of need. We correlate each allocation with each reading, over all applicants and
among the funded ones. The highest correlation reveals the reading the group most likely used.

In [ ]:
READINGS = ["need_form_x12", "shortfall_csv", "shortfall_text"]


def need_reading(a):
    f = a > 0
    out = {r: spearmanr(a, F[r])[0] for r in READINGS}
    out.update({r + " (among funded)": spearmanr(a[f], F.loc[f, r])[0] for r in READINGS})
    return pd.Series(out)


show_table(pd.DataFrame({"own": need_reading(OWN), "other": need_reading(OTHER)}).round(2), "Rank correlation with each reading of need")

**How to read it.** Each row is one definition of need. Compare the rows within a column. If your allocation
correlates most with `shortfall_text`, your group's notion of need matches what the forms document for the grant
year; if the other group correlates most with `need_form_x12`, it took the form's current monthly gap at face value.

## 6.2 Criteria profile and trade-offs

We score each allocation on eight indicators between 0 and 1, where higher means "more of this value":

| Indicator | Meaning | Link to the call |
|---|---|---|
| need targeting | share of the budget reaching the third of applicants with the largest documented shortfall | Financial need (40%) |
| depth | average share of the documented ceiling paid to funded students | Financial need (40%) |
| breadth | share of applicants with a documented shortfall who receive something | Equal opportunity |
| academic | budget-weighted average GPA rank of recipients | Academic potential (25%) |
| resilience | budget-weighted share of resilience boxes ticked | Resilience (20%) |
| diversity | budget share to first-generation students or students from low-income countries | Diversity (10%) |
| rule conformity | share of the budget within documented ceilings and outside the "condition" rows of the audit | Eligibility, award rules |
| equality | 1 − Gini among recipients | Equal treatment |

With a fixed budget, no allocation can maximise all eight: more depth costs breadth, and academic selection usually
moves money away from the neediest. One allocation **dominates** the other only if it is at least as good on every
indicator. Otherwise the groups made a **trade-off**, and the indicators on which each is ahead describe it.

In [ ]:
INDICATORS = ["need targeting", "depth", "breadth", "academic", "resilience", "diversity", "rule conformity", "equality"]
NEEDS_REASON = ((F["permitC_family_resources"] == "rule applies") | (F["doctoral_funding_documented"] == 0)
                | (F["low_gpa_status"] == "not explained") | (F["study_permit_conflict"] == 1))


def criteria_profile(a):
    w, f, has = a / a.sum(), a > 0, CEILING > 0
    top_third = F["shortfall_text"] >= F["shortfall_text"].quantile(2 / 3)
    conforming = np.minimum(a, CEILING).where(~NEEDS_REASON, 0)
    return pd.Series({
        "need targeting": w[top_third].sum(),
        "depth": (a[f & has] / CEILING[f & has]).clip(upper=1).mean() if (f & has).any() else 0,
        "breadth": (f & has).sum() / has.sum(),
        "academic": (w * CSV["gpa"].rank(pct=True)).sum(),
        "resilience": (w * CSV[RESILIENCE_FACTORS].sum(axis=1) / 5).sum(),
        "diversity": w[(CSV["risk_first_generation"] == 1) | (CSV["gni_tier"] == 1)].sum(),
        "rule conformity": conforming.sum() / a.sum(),
        "equality": 1 - gini(a[f]),
    })[INDICATORS]


def trade_off(pa, pb, tol=0.02):
    ahead_a = [i for i in INDICATORS if pa[i] - pb[i] > tol]
    ahead_b = [i for i in INDICATORS if pb[i] - pa[i] > tol]
    verdict = ("own dominates other" if ahead_a and not ahead_b else "other dominates own" if ahead_b and not ahead_a
               else "no difference" if not (ahead_a or ahead_b) else "trade-off")
    return verdict, ahead_a, ahead_b


PROF_OWN, PROF_OTHER = criteria_profile(OWN), criteria_profile(OTHER)
show_table(pd.DataFrame({"own": PROF_OWN, "other": PROF_OTHER, "own − other": PROF_OWN - PROF_OTHER}).round(2), "Criteria profile")
VERDICT, AHEAD_OWN, AHEAD_OTHER = trade_off(PROF_OWN, PROF_OTHER)
print(f"Verdict: {VERDICT}.  Own ahead on: {AHEAD_OWN or 'nothing'}.  Other ahead on: {AHEAD_OTHER or 'nothing'}.")

ang = np.linspace(0, 2 * np.pi, len(INDICATORS), endpoint=False).tolist()
fig, ax = plt.subplots(figsize=(4.8, 4.8), subplot_kw={"polar": True})
for prof, c, lab in [(PROF_OWN, OWN_COLOR, "own"), (PROF_OTHER, OTHER_COLOR, "other")]:
    v = prof.tolist()
    ax.plot(ang + ang[:1], v + v[:1], color=c, lw=2, label=lab); ax.fill(ang + ang[:1], v + v[:1], color=c, alpha=0.15)
ax.set_xticks(ang); ax.set_xticklabels(INDICATORS, fontsize=8); ax.set_ylim(0, 1); ax.set_rlabel_position(22.5)
ax.tick_params(axis="y", labelsize=6, colors="grey")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), frameon=False); show_figure("6.2 Criteria profile (radar)")

**How to read it.** Where one polygon reaches further out, that allocation gives more weight to the value on that
axis. A trade-off cannot be settled with data; it has to be argued. The verdict tells you which values your group
gave up for which others.

---
# Part 7. Do characteristics that no criterion justifies influence the awards?

The call's equal-opportunity policy excludes gender as a ground for decisions, and no criterion mentions age or the
length of the statements. An allocation can still depend on them, by accident or through a biased step: labels
written by tired readers, an LLM's verbosity bias (2.15), a proxy hidden in another variable.

**Method, in two moves.** First, a flexible model (gradient boosting) predicts each amount from the *criteria only*:
the CSV criteria and the form-based criteria of Part 4.1. Each applicant is predicted by a model that has not seen
that applicant (cross-validation), so the model cannot memorise the amounts. Second, we compare the part of the
amount that the criteria do *not* explain between groups of applicants. A large and statistically reliable gap means
that the characteristic influences the award beyond the criteria.

We also include one criterion-related probe on purpose: applicants who *state* that others need the grant more
(2.13). Candour should not cost money once need is accounted for; if it does, the allocation penalises honesty.

In [ ]:
CRIT_COLS = [c for c in X.columns if FEATURE_KIND[c] != "not a criterion" and c != "accepts_lower_priority"]
PROBES = {"gender: female vs male": (CSV["gender"] == "Female", CSV["gender"] == "Male"),
          "age: 30 or older vs under 30": (CSV["age"] >= 30, CSV["age"] < 30),
          "statements: longest third vs shortest third": (F["statement_words"] >= F["statement_words"].quantile(2 / 3),
                                                          F["statement_words"] <= F["statement_words"].quantile(1 / 3)),
          "candour: accepts lower priority vs not": (F["accepts_lower_priority"] == 1, F["accepts_lower_priority"] == 0)}


def unexplained_gaps(a):
    model = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=0)
    resid = a - cross_val_predict(model, X[CRIT_COLS], a, cv=5)
    return pd.DataFrame({name: {"gap (CHF)": resid[g1].mean() - resid[g0].mean(),
                                "p-value": ttest_ind(resid[g1], resid[g0], equal_var=False)[1]}
                         for name, (g1, g0) in PROBES.items()}).T


GAPS = pd.concat({"own": unexplained_gaps(OWN), "other": unexplained_gaps(OTHER)}, axis=1)
show_table(GAPS.round(3), "Average award difference that the criteria do not explain")

**How to read it.** `gap (CHF)` is the average extra amount the first group receives compared with the second, after
accounting for the criteria. The p-value states how often a gap of this size would arise by chance alone. We run
four probes per allocation, so a single small p-value can appear by chance; take a gap seriously only when p is
below 0.01 **and** the gap is large enough to matter (say, CHF 150 or more). Even then, a gap is a question for the
other group, not a verdict: a characteristic can correlate with a consideration our criteria do not capture.

---
# Part 8. The four graded dimensions of the comparison

The grading grid of the comparison report names four dimensions: **mission alignment**, **fairness and distributive
justice**, **population diversity and representation**, and **bias and disparate impact**. Parts 1 to 7 contain
evidence for each of them, spread over many tables. This part gathers, for each dimension, the indicators that bear
on it most directly, computed in the same way for both allocations, and adds the analyses that were still missing.

None of these indicators is a grade. Each dimension asks a question that data alone cannot settle: *which*
principle of justice, *which* groups should be represented, *when* a gap is a bias. The indicators tell you what each
allocation does; your report must say whether that is right, by the standard of the call and of the foundation's
mission (equal opportunity for international students in financial need).

## 8.1 Mission alignment: does the money go where the call says it should?

**Why these indicators.** The call funds the documented shortfall of eligible students for September 2027 to August
2028, and asks the committee to weigh circumstances that only the forms reveal. Three questions follow. (a) How much
of each franc closes a documented gap, and how much goes above it or to applicants whose eligibility needs a reason?
(b) How much of the pool's documented need does the budget close, and for how many applicants? (c) Does the
allocation reach the applicants whose life events the call asks the committee to weigh (family support at risk or
withdrawn, a scholarship that ends, the Permit C exception)? We add the distance between the weights each allocation
gives the criteria and the call's weights (4.8).

In [ ]:
NEEDY = CEILING > 0
PENDING = F["permit_pending"] == 1
LIFE_EVENTS = ((F["permitC_family_resources"] == "exception candidate") | (F["family_support_at_risk"] == 1)
               | (F["family_support_withdrawn"] == 1) | (F["scholarship_status"] == "ends before grant year"))


def mission(a, lab):
    within = np.minimum(a, CEILING)
    return pd.Series({
        "share of budget within documented need": within.sum() / a.sum(),
        "share of budget above the documented ceiling": (a - CEILING).clip(lower=0).sum() / a.sum(),
        "share of budget to applicants without a documented shortfall": a[~NEEDY].sum() / a.sum(),
        "share of budget whose eligibility needs a stated reason": a[NEEDS_REASON].sum() / a.sum(),
        "share of budget conditional on a pending permit": a[PENDING].sum() / a.sum(),
        "share of the pool's documented need closed": within.sum() / CEILING.sum(),
        "share of applicants with documented need reached": ((a > 0) & NEEDY).sum() / NEEDY.sum(),
        "share of life-event cases with need reached": ((a > 0) & LIFE_EVENTS & NEEDY).sum() / max(1, (LIFE_EVENTS & NEEDY).sum()),
        "distance of criterion weights from the call (4.8)": DIST_TO_CALL[lab],
    })


MISSION = pd.DataFrame({"own": mission(OWN, "own"), "other": mission(OTHER, "other")})
print(f"The pool's documented need for the grant year is CHF {CEILING.sum():,.0f} for {int(NEEDY.sum())} applicants; "
      f"the budget can close at most {BUDGET / CEILING.sum():.0%} of it.")
show_table(MISSION, "8.1 Mission alignment")


def franc_split(a):
    within = np.minimum(a, CEILING)
    return pd.Series({"within documented need, no condition": within[~NEEDS_REASON & ~PENDING].sum(),
                      "within documented need, conditional on a pending permit": within[~NEEDS_REASON & PENDING].sum(),
                      "within documented need, eligibility needs a reason": within[NEEDS_REASON].sum(),
                      "above the documented ceiling (or no documented gap)": (a - CEILING).clip(lower=0).sum()})


SPLIT = pd.DataFrame({"own": franc_split(OWN), "other": franc_split(OTHER)})
fig, ax = plt.subplots(figsize=(10, 2.6))
left = np.zeros(2)
for (cat, row), c in zip(SPLIT.iterrows(), [OWN_COLOR, "#4a7fb5", ACCENT, "#c0392b"]):
    ax.barh(["own", "other"], row.values, left=left, color=c, label=cat)
    for i, v in enumerate(row.values):
        if v > 15000:
            ax.text(left[i] + v / 2, i, f"{v / 1000:,.0f}k", ha="center", va="center", color="white", fontsize=8)
    left += row.values
ax.set(xlabel="CHF", title="Where each franc goes"); ax.invert_yaxis()
ax.legend(frameon=False, fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.35))
show_figure("8.1 Where each franc goes")

**How to read it.** The dark segment is money that closes a documented gap of an applicant whose eligibility raises
no question: the core of the mission. The orange segment closes documented gaps of applicants whose eligibility the
call allows only with a reason (Permit C with family resources, doctoral gap not documented, low GPA not explained,
study-permit conflict); the red segment goes beyond what the forms document. Neither is an error by itself: an
allocation may pay above the ceiling on purpose (for example, if a group judged that the forms understate costs), and
a grey-zone award can be the right decision. But each franc in these segments needs a sentence in the report.

**The human question.** Which allocation serves equal opportunity for students in need better, and on what reading
of the call? A high share within need with few applicants reached, and a lower share with many reached, are two
defensible readings; say which one your group chose, and why.

## 8.2 Fairness and distributive justice: which principle does each allocation follow?

**Why these indicators.** Theories of distributive justice disagree about what a fair distribution is, so we measure
each allocation against several principles rather than one:

| Principle | Question | Indicator |
|---|---|---|
| **need** (vertical equity) | Do those with larger needs receive more? | ρ between award and documented ceiling among applicants with need; concentration index (Wagstaff, Paci and van Doorslaer, 1991) |
| **equality** | Do recipients receive similar amounts? | 1 − Gini among recipients |
| **priority to the worst-off** (Parfit, 1997) | Are the most disadvantaged served first? | share of the worst-off fifth funded, and share of their gap closed |
| **sufficiency** (Frankfurt, 1987) | How many are brought to "enough"? | share of applicants with need whose remaining gap is below `SUFFICIENCY_GAP_CHF` |
| **horizontal equity** (like cases alike) | Are similar applicants treated similarly? | consistency of decisions among nearest neighbours (Zemel et al., 2013) |
| **desert** | Do better students receive more? | ρ between award and GPA among applicants with need |

The *worst-off* are the fifth of applicants with documented need who rank highest on a simple deprivation index:
the average of the rank of the documented shortfall and the rank of the number of substantiated adversities (specific
risk boxes, family support at risk or withdrawn). *Neighbours* are the `SIMILAR_NEIGHBOURS` applicants closest on the
standardised criteria of Part 7; treating every criterion as equally important in this distance is itself a choice.
The **concentration index** lies between −1 and 1: positive values mean the budget is concentrated on applicants
with larger documented needs, 0 means no relation.

In [ ]:
DEPRIVATION = 0.5 * F["shortfall_text"].rank(pct=True) + 0.5 * (
    F["risk_ticks_specific"] + F["family_support_at_risk"] + F["family_support_withdrawn"]).rank(pct=True)
WORST_OFF = NEEDY & (DEPRIVATION >= DEPRIVATION[NEEDY].quantile(0.8))
Z_CRIT = pd.DataFrame(StandardScaler().fit_transform(X[CRIT_COLS]), index=X.index, columns=CRIT_COLS)
NN_IDX = NearestNeighbors(n_neighbors=SIMILAR_NEIGHBOURS + 1).fit(Z_CRIT).kneighbors(Z_CRIT)[1][:, 1:]
NEED_RANK = CEILING.rank(pct=True, method="average")


def concentration_index(a):
    """2 cov(award, need rank) / mean award: > 0 means awards concentrated on larger documented needs."""
    return 2 * np.cov(a, NEED_RANK, bias=True)[0, 1] / a.mean()


def justice(a):
    f = (a > 0).astype(int).values
    remaining = (CEILING - np.minimum(a, CEILING))[NEEDY]
    return pd.Series({
        "need: ρ award vs documented ceiling (applicants with need)": spearmanr(a[NEEDY], CEILING[NEEDY])[0],
        "need: concentration index": concentration_index(a),
        "equality: 1 − Gini among recipients": 1 - gini(a[a > 0]),
        "priority: share of the worst-off fifth funded": (a[WORST_OFF] > 0).mean(),
        "priority: share of the worst-off fifth's gap closed": np.minimum(a, CEILING)[WORST_OFF].sum() / CEILING[WORST_OFF].sum(),
        f"sufficiency: share of applicants with need left with < CHF {SUFFICIENCY_GAP_CHF:,}": (remaining < SUFFICIENCY_GAP_CHF).mean(),
        "sufficiency: share of applicants with need who receive at least half their gap":   # CHF 50 tolerance for rounding
            (np.minimum(a, CEILING)[NEEDY] + 50 >= 0.5 * CEILING[NEEDY]).mean(),
        "horizontal equity: consistency of decisions among neighbours": 1 - np.abs(f[:, None] - f[NN_IDX]).mean(),
        "horizontal equity: mean award difference to neighbours (CHF)": np.abs(a.values[:, None] - a.values[NN_IDX]).mean(),
        "desert: ρ award vs GPA (applicants with need)": spearmanr(a[NEEDY], CSV.loc[NEEDY, "gpa"])[0],
    })


JUSTICE = pd.DataFrame({"own": justice(OWN), "other": justice(OTHER)})
show_table(JUSTICE, "8.2 Distributive justice: each allocation against six principles")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = CEILING.sort_values(ascending=False).index
xs = np.linspace(0, 1, len(order))
ax[0].plot(xs, np.cumsum(CEILING[order]) / CEILING.sum(), color="black", lw=1, ls=":", label="proportional to documented need")
ax[0].plot([0, 1], [0, 1], color=GREY, lw=0.8, ls="--", label="same amount for everyone")
for a, c, lab in [(OWN, OWN_COLOR, "own"), (OTHER, OTHER_COLOR, "other")]:
    ax[0].plot(xs, np.cumsum(a[order]) / a.sum(), color=c, lw=2, label=f"{lab} (CI {concentration_index(a):+.2f})")
ax[0].set(xlabel="applicants, largest documented ceiling first", ylabel="cumulative share of the budget",
          title="Concentration curves: who receives the money?")
ax[0].legend(frameon=False, fontsize=7)
bins = np.linspace(0, 1, 11)
for a, c, lab in [(OWN, OWN_COLOR, "own"), (OTHER, OTHER_COLOR, "other")]:
    closed = (np.minimum(a, CEILING) / CEILING.where(NEEDY))[NEEDY]
    ax[1].hist(closed, bins=bins, histtype="step", lw=2, color=c, label=lab)
ax[1].set(xlabel="share of the documented gap closed (0 = nothing, 1 = fully)", ylabel="applicants with documented need",
          title="How much of each applicant's gap is closed?")
ax[1].legend(frameon=False)
show_figure("8.2 Concentration curves and gap closure")


def unequal_neighbours(a, k=4):
    rows, seen = [], set()
    d = np.abs(a.values[:, None] - a.values[NN_IDX])
    for i, j in sorted(((i, j) for i in range(len(a)) for j in range(NN_IDX.shape[1])), key=lambda t: -d[t])[:200]:
        s1, s2 = a.index[i], a.index[NN_IDX[i, j]]
        if s1 in seen or s2 in seen:              # show each applicant once
            continue
        seen.update((s1, s2))
        diffs = (Z_CRIT.loc[s1] - Z_CRIT.loc[s2]).abs().sort_values(ascending=False).head(2).index
        rows.append({"applicant": s1, "award": a[s1], "nearest neighbour": s2, "neighbour's award": a[s2],
                     "where they differ most": "; ".join(f"{c}: {X.at[s1, c]:g} vs {X.at[s2, c]:g}" for c in diffs)})
        if len(rows) == k:
            break
    return pd.DataFrame(rows).set_index("applicant")


show_table(unequal_neighbours(OWN), "Similar applicants, very different awards: own allocation")
show_table(unequal_neighbours(OTHER), "Similar applicants, very different awards: other allocation")

**How to read it.** In the left panel, a curve above the dotted line concentrates the budget even more on large
needs than a need-proportional rule would; a curve close to the grey diagonal pays every applicant the same,
regardless of need. In the right panel, a peak at 0 and a peak at 1 mean "few, fully funded"; a peak in the middle
means "many, partly funded". The two tables show pairs of applicants who are close on every criterion but receive
very different awards: open both forms and decide whether the difference is justified. Such pairs are the most
concrete test of horizontal equity, and the hardest to defend when nothing in the forms explains them.

**The human question.** Which principle did each group follow (need, equality, priority to the worst-off,
sufficiency, merit)? Is it defensible for this foundation? No allocation can lead on all six principles with a
fixed budget; name what yours gives up.

## 8.3 Population diversity and representation: who is among the recipients, and who is missing?

**Why these indicators.** Representation compares the composition of the recipients with a reference population.
The choice of reference matters. Compared with *all applicants*, a group is under-represented if it receives fewer
awards than its size; but if the group has less documented need, that is what a need-based call intends. Compared
with *applicants with documented need*, under-representation is harder to explain by the call. We therefore report
the **representation ratio** against both references: share among recipients divided by share in the reference (1 =
proportional; 0.5 = half as many as proportional). We also measure how varied the recipients are, with Shannon's
diversity index converted to an *effective number of groups* (exp of the index; Jost, 2006) and to *evenness*
(Pielou, 1966: 1 = all groups equally represented).

In [ ]:
REP_DIMENSIONS = {
    "geographic region": CSV["geographic_region"], "study level": CSV["study_level"], "gender": CSV["gender"],
    "GNI tier": CSV["gni_tier"].map({1: "tier 1 (lowest income)", 2: "tier 2", 3: "tier 3"}), "residence permit": CSV["residency_permit"],
    "study mode": CSV["study_mode"],
    "circumstance": None,
}
CIRCUMSTANCES = {"first-generation": CSV["risk_first_generation"] == 1, "refugee or asylum": CSV["risk_refugee_asylum"] == 1,
                 "disability": CSV["risk_disability"] == 1, "housing insecurity": CSV["risk_housing_insecurity"] == 1,
                 "parent student": CSV["risk_parent_student"] == 1}


def representation():
    rows = []
    for dim, g in REP_DIMENSIONS.items():
        groups = CIRCUMSTANCES.items() if g is None else ((v, g == v) for v in g.value_counts().index)
        for name, m in groups:
            row = {"dimension": dim, "group": name, "applicants": int(m.sum()), "with documented need": int((m & NEEDY).sum())}
            for lab, a in [("own", OWN), ("other", OTHER)]:
                f = a > 0
                share_rec = (f & m).sum() / f.sum()
                row[f"{lab}: recipients"] = int((f & m).sum())
                row[f"{lab}: ratio vs applicants"] = share_rec / (m.mean())
                row[f"{lab}: ratio vs needy"] = share_rec / ((m & NEEDY).sum() / NEEDY.sum()) if (m & NEEDY).sum() else np.nan
                row[f"{lab}: budget ratio vs needy"] = (a[m].sum() / a.sum()) / ((m & NEEDY).sum() / NEEDY.sum()) if (m & NEEDY).sum() else np.nan
            rows.append(row)
    return pd.DataFrame(rows).set_index(["dimension", "group"])


REPRESENTATION = representation()
show_table(REPRESENTATION, "8.3 Representation ratios (1 = proportional)",
           gradient=["own: ratio vs needy", "other: ratio vs needy"])

hm = REPRESENTATION[REPRESENTATION["with documented need"] >= MIN_GROUP_SIZE][["own: ratio vs needy", "other: ratio vs needy"]]
fig, ax = plt.subplots(figsize=(6.2, 0.28 * len(hm) + 1.2))
im = ax.imshow(np.log2(hm.clip(lower=0.25, upper=4).values), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["own", "other"])
ax.set_yticks(range(len(hm))); ax.set_yticklabels([f"{d}: {g}" for d, g in hm.index], fontsize=7)
for (i, j), v in np.ndenumerate(hm.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Representation among recipients, relative to applicants with need\n(blue = over-, red = under-represented)", fontsize=9)
plt.colorbar(im, ax=ax, ticks=[-2, -1, 0, 1, 2], format=lambda v, _: f"×{2.0 ** v:g}", fraction=0.05)
show_figure("8.3 Representation heatmap")


def diversity_of(a):
    f = a > 0
    out = {}
    for name, col in [("region", "geographic_region"), ("country", "country_origin"), ("field of study", "field_of_study"),
                      ("university", "university"), ("study level", "study_level")]:
        counts = CSV.loc[f, col].value_counts()
        h = entropy(counts)
        out[f"{name}: groups represented"] = f"{len(counts)} of {CSV[col].nunique()}"
        out[f"{name}: effective number of groups"] = np.exp(h)
        out[f"{name}: evenness"] = h / np.log(CSV[col].nunique())
    return pd.Series(out)


DIVERSITY = pd.DataFrame({"applicants": diversity_of(pd.Series(1, index=CSV.index)), "own": diversity_of(OWN), "other": diversity_of(OTHER)})
show_table(DIVERSITY, "Diversity of the recipients")

INTERSECTIONS = {
    "first-generation and from a tier-1 country": (CSV["risk_first_generation"] == 1) & (CSV["gni_tier"] == 1),
    "women in doctoral programmes": (CSV["gender"] == "Female") & (CSV["study_level"] == "Doctoral"),
    "non-binary or gender not stated": CSV["gender"].isin(["Non-binary", "Prefer not to say"]),
    "refugee or asylum background, part-time": (CSV["risk_refugee_asylum"] == 1) & (CSV["study_mode"] == "Part-time"),
    "disability and first-generation": (CSV["risk_disability"] == 1) & (CSV["risk_first_generation"] == 1),
}
show_table(pd.DataFrame({name: {"applicants": int(m.sum()), "with documented need": int((m & NEEDY).sum()),
                                "own: funded": int((OWN[m] > 0).sum()), "other: funded": int((OTHER[m] > 0).sum()),
                                "own: CHF": OWN[m].sum(), "other: CHF": OTHER[m].sum()} for name, m in INTERSECTIONS.items()}).T,
           "Intersections: groups defined by two characteristics")
MISSING = {lab: [f"{d}: {g} ({r[lab + ': ratio vs needy']:.2f})" for (d, g), r in REPRESENTATION.iterrows()
                 if r["with documented need"] >= MIN_GROUP_SIZE and r[lab + ": ratio vs needy"] < 0.5] for lab in ["own", "other"]}
for lab, groups in MISSING.items():
    print(f"{lab}: groups with at least {MIN_GROUP_SIZE} needy applicants and less than half their proportional share of awards: "
          + ("; ".join(groups) or "none"))

**How to read it.** In the heatmap, a red cell for your allocation and a white cell for the other is a group your
allocation leaves out more than the other does. Check the reason: a group can be under-represented because of a
screen (5.1), for example pending permits, or as a side effect, for example doctoral students with smaller
ceilings in an allocation that ranks by CHF of need. The diversity table tells you whether the recipients are drawn
from many or few regions, countries, fields and universities; a lower effective number than the applicant pool means
concentration. Representation of gender is reported here for completeness; whether it is *caused* by gender is the
question of 8.4.

**The human question.** Who is represented among the recipients, who is missing, and does it matter for the mission?
The call gives diversity 10% of the weight; decide which groups your group considers essential to represent and say
whether either allocation achieves it.

## 8.4 Bias and disparate impact: do characteristics that are not criteria shape the outcome?

**Why these indicators.** Part 7 tested whether characteristics without a criterion (gender, age, length of the
statements) influence the *amounts* beyond the criteria. Here we look at the decision to fund, with three tools.

1. **Adverse impact ratio.** The selection rate of a group divided by the rate of the most selected group of the same
   characteristic. US employment guidelines treat a ratio below 0.8 as a sign of adverse impact (the "four-fifths
   rule", Uniform Guidelines on Employee Selection Procedures, 1978). It ignores need entirely, so it flags
   differences that the call may justify.
2. **Need-adjusted ratio (observed / expected).** A classifier predicts each applicant's chance of being funded *from
   the criteria only*, without seeing the applicant (cross-validation). Summing these chances over a group gives the
   number of recipients expected if the group were treated like comparable applicants. Observed divided by expected
   below 0.8 or above 1.25 means the group receives fewer or more awards than its criteria predict.
3. **Counterfactual flip and proxy check.** We change one characteristic in the surrogate of 4.1 (women ↔ men; age set
   to the median; statement length set to the median) and measure how much the predicted amounts move. This shows
   whether the surrogate *uses* the characteristic; it is not a causal test (Kusner et al., 2017). Finally, we ask how
   well the criteria predict each characteristic (AUC): if they predict it well, a gap can arise through *proxies*
   even when nobody looked at the characteristic.

Small groups produce ratios far from 1 by chance alone. A ratio is therefore **flagged** only when chance is an
unlikely explanation: the adverse impact ratio when Fisher's exact test gives p < 0.05, the observed / expected ratio
when observed and expected differ by more than two standard deviations. Groups with fewer than `MIN_GROUP_SIZE`
applicants are not tested. Region is included with a warning: the diversity criterion can legitimately favour regions
that are rare among applicants.

In [ ]:
clf = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=0)
EXPECTED = {lab: pd.Series(cross_val_predict(clf, X[CRIT_COLS], (a > 0).astype(int), cv=5, method="predict_proba")[:, 1], index=X.index)
            for lab, a in [("own", OWN), ("other", OTHER)]}
AGE_BAND = pd.cut(CSV["age"], [0, 24, 29, 120], labels=["under 25", "25 to 29", "30 or older"]).astype(str)
LONG_TEXT = pd.cut(F["statement_words"].rank(pct=True), [0, 1 / 3, 2 / 3, 1], labels=["short third", "middle third", "long third"]).astype(str)
BIAS_DIMENSIONS = {"gender": CSV["gender"], "age": AGE_BAND, "statement length": LONG_TEXT, "university": CSV["university"],
                   "field of study": CSV["field_of_study"], "region (may be intended: diversity)": CSV["geographic_region"]}


def impact():
    rows = []
    for dim, g in BIAS_DIMENSIONS.items():
        valid = g.value_counts()[lambda s: s >= MIN_GROUP_SIZE].index
        for lab, a in [("own", OWN), ("other", OTHER)]:
            f = a > 0
            rates = f.groupby(g).mean()[valid]
            for v in valid:
                m = g == v
                obs, exp = f[m].sum(), EXPECTED[lab][m].sum()
                z = (obs - exp) / np.sqrt(max((EXPECTED[lab][m] * (1 - EXPECTED[lab][m])).sum(), 1e-9))
                p_rate = fisher_exact([[int(f[m].sum()), int((~f[m]).sum())], [int(f[~m].sum()), int((~f[~m]).sum())]])[1]
                air, oe = (rates[v] / rates.max() if rates.max() > 0 else np.nan), obs / max(exp, 1e-9)
                flag = ("AIR<0.8 " if air < 0.8 and p_rate < 0.05 else "") + \
                       ("O/E<0.8" if oe < 0.8 and z < -2 else "O/E>1.25" if oe > 1.25 and z > 2 else "")
                rows.append({"characteristic": dim, "group": v, "allocation": lab, "applicants": int(m.sum()), "funded rate": rates[v],
                             "adverse impact ratio": air, "observed / expected": oe, "flag": flag.strip()})
    t = pd.DataFrame(rows)
    wide = t.pivot_table(index=["characteristic", "group"], columns="allocation", aggfunc="first",
                         values=["applicants", "funded rate", "adverse impact ratio", "observed / expected", "flag"])
    wide.columns = [f"{al}: {m}" for m, al in wide.columns]
    wide = wide.rename(columns={"own: applicants": "applicants"})
    return wide[["applicants"] + [f"{al}: {m}" for al in ["own", "other"]
                                  for m in ["funded rate", "adverse impact ratio", "observed / expected", "flag"]]]


IMPACT = impact()
show_table(IMPACT, "8.4 Disparate impact on the decision to fund",
           gradient=["own: observed / expected", "other: observed / expected"])

sub = IMPACT[IMPACT.index.get_level_values(0).isin(["gender", "age", "statement length", "region (may be intended: diversity)"])]
fig, ax = plt.subplots(figsize=(7.5, 0.3 * len(sub) + 1.2))
y = np.arange(len(sub))
ax.axvspan(0.8, 1.25, color=GREY, alpha=0.25, label="0.8 to 1.25")
ax.scatter(sub["own: observed / expected"], y, color=OWN_COLOR, label="own", zorder=3)
ax.scatter(sub["other: observed / expected"], y, color=OTHER_COLOR, label="other", zorder=3)
ax.axvline(1, color="black", lw=0.6)
ax.set_yticks(y); ax.set_yticklabels([f"{d.split(' (')[0]}: {g}" for d, g in sub.index], fontsize=7); ax.invert_yaxis()
ax.set(xlabel="awards observed / awards expected from the criteria", title="Need-adjusted funding ratio by group")
ax.legend(frameon=False, fontsize=7)
show_figure("8.4 Need-adjusted funding ratio by group")


def flip_tests(sur):
    base = sur["gbm"].predict(X)
    variants = {}
    Xg = X.copy(); Xg[["gender_female", "gender_male"]] = X[["gender_male", "gender_female"]].values
    variants["women ↔ men"] = Xg
    Xa = X.copy(); Xa["age"] = X["age"].median(); variants["age set to the median"] = Xa
    Xs = X.copy(); Xs["statement_words"] = X["statement_words"].median(); variants["statement length set to the median"] = Xs
    out = {}
    for name, Xv in variants.items():
        d = sur["gbm"].predict(Xv) - base
        out[name] = {"mean |change| (CHF)": np.abs(d).mean(), "applicants moved by > CHF 200": int((np.abs(d) > 200).sum())}
    return pd.DataFrame(out).T


FLIP = pd.concat({"own": flip_tests(S_OWN), "other": flip_tests(S_OTHER)}, axis=1)
show_table(FLIP, "Counterfactual flip test on the surrogates")


def proxy_auc(target):
    m = target.notna()
    p = cross_val_predict(GradientBoostingClassifier(random_state=0), X.loc[m, CRIT_COLS], target[m].astype(int), cv=5, method="predict_proba")[:, 1]
    return roc_auc_score(target[m].astype(int), p)


PROXY = pd.Series({"female (vs male)": proxy_auc((CSV["gender"] == "Female").where(CSV["gender"].isin(["Female", "Male"]))),
                   "30 or older": proxy_auc(CSV["age"] >= 30),
                   "long statements (top third)": proxy_auc(F["statement_words"] >= F["statement_words"].quantile(2 / 3))},
                  name="AUC from the criteria").to_frame()
show_table(PROXY, "Proxy check: how well do the criteria predict each characteristic? (0.5 = not at all)")

**How to read it.** Start with the *observed / expected* columns: they compare each group with applicants who have
the same criteria, which is closer to the question of bias than the raw rates. A flag on one allocation and not on
the other is the most informative pattern, because both allocations face the same applicants. With about 45 groups
tested per allocation, two or three flags are expected by chance alone even when nothing is wrong: treat an isolated
flag on a small group (a single field of study or university) as a question, not a finding. Then look at the flip
test: if switching women and men barely moves the surrogate's amounts, a gap in 8.4 or Part 7 does not come from
gender directly; if the proxy check shows that the criteria predict gender or age well (AUC above about 0.7), it may
come from a criterion correlated with them. Statement length deserves particular attention when a group used an LLM
or human readers who may reward long texts.

**The human question.** Is each gap a bias, a proxy or chance? Consider the size of the group, the p-values of
Part 7, and whether a criterion of the call explains the gap. What would you ask the other group, and what would you
change in your own procedure?

## 8.5 Scorecard of the four dimensions

The table gathers the indicators of 8.1 to 8.4 that are most useful for the report. For each indicator, the column
*the call favours* says whether the call itself indicates a direction (for example, less money above documented
ceilings). Where the answer is *a matter of principle*, the call does not settle it and the report must argue it.

In [ ]:
def worst(series_name, lab):
    v = IMPACT[f"{lab}: observed / expected"]
    return v[IMPACT.index.get_level_values(0) == series_name]


SCORECARD_ROWS = [
    ("mission", "share of budget within documented need", MISSION.loc["share of budget within documented need"], "higher"),
    ("mission", "share of budget above the documented ceiling", MISSION.loc["share of budget above the documented ceiling"], "lower"),
    ("mission", "share of budget whose eligibility needs a stated reason", MISSION.loc["share of budget whose eligibility needs a stated reason"], "lower, unless each case is argued"),
    ("mission", "share of the pool's documented need closed", MISSION.loc["share of the pool's documented need closed"], "higher"),
    ("mission", "share of life-event cases with need reached", MISSION.loc["share of life-event cases with need reached"], "higher"),
    ("mission", "distance of criterion weights from the call", MISSION.loc["distance of criterion weights from the call (4.8)"], "lower"),
    ("justice", "need: concentration index", JUSTICE.loc["need: concentration index"], "higher (need weighs 40%)"),
    ("justice", "equality: 1 − Gini among recipients", JUSTICE.loc["equality: 1 − Gini among recipients"], "a matter of principle"),
    ("justice", "priority: share of the worst-off fifth funded", JUSTICE.loc["priority: share of the worst-off fifth funded"], "a matter of principle"),
    ("justice", "sufficiency: applicants with need who receive at least half their gap",
     JUSTICE.loc["sufficiency: share of applicants with need who receive at least half their gap"], "a matter of principle"),
    ("justice", "horizontal equity: consistency among neighbours", JUSTICE.loc["horizontal equity: consistency of decisions among neighbours"], "higher"),
    ("diversity", "regions: effective number among recipients", DIVERSITY.loc["region: effective number of groups", ["own", "other"]].astype(float), "a matter of principle"),
    ("diversity", "groups with less than half their proportional share", pd.Series({k: len(v) for k, v in MISSING.items()}), "lower"),
    ("bias", "gender: lowest observed / expected", pd.Series({lab: worst("gender", lab).min() for lab in ["own", "other"]}), "closer to 1"),
    ("bias", "age: lowest observed / expected", pd.Series({lab: worst("age", lab).min() for lab in ["own", "other"]}), "closer to 1"),
    ("bias", "statement length: highest observed / expected", pd.Series({lab: worst("statement length", lab).max() for lab in ["own", "other"]}), "closer to 1"),
]


def closer(own, other, direction, tol=0.01):
    if abs(own - other) <= tol * max(1, abs(own), abs(other)):
        return "equal"
    if direction.startswith("higher"):
        return "own" if own > other else "other"
    if direction.startswith("lower"):
        return "own" if own < other else "other"
    if direction == "closer to 1":
        return "own" if abs(own - 1) < abs(other - 1) else "other"
    return "—"


SCORECARD = pd.DataFrame([{"dimension": d, "indicator": i, "own": float(v["own"]), "other": float(v["other"]), "the call favours": dirn,
                           "closer to the call": closer(float(v["own"]), float(v["other"]), dirn)}
                          for d, i, v, dirn in SCORECARD_ROWS]).set_index(["dimension", "indicator"])
show_table(SCORECARD, "8.5 Scorecard of the four graded dimensions")

shares = SCORECARD[(SCORECARD[["own", "other"]].abs() <= 1).all(axis=1)]
fig, ax = plt.subplots(figsize=(8.5, 0.34 * len(shares) + 1))
y = np.arange(len(shares))
for i, (_, r) in enumerate(shares.iterrows()):
    ax.plot([r["own"], r["other"]], [i, i], color=GREY, lw=2, zorder=1)
ax.scatter(shares["own"], y, color=OWN_COLOR, s=40, label="own", zorder=2)
ax.scatter(shares["other"], y, color=OTHER_COLOR, s=40, label="other", zorder=2)
ax.set_yticks(y); ax.set_yticklabels([f"[{d}] {i}" for d, i in shares.index], fontsize=7); ax.invert_yaxis()
ax.set(xlabel="value", title="The two allocations on the scorecard indicators measured between 0 and 1")
ax.legend(frameon=False, fontsize=8)
show_figure("8.5 Scorecard dumbbell chart")

**How to read it.** Count neither wins nor losses: the indicators are not equally important and several measure
the same value twice. Use the scorecard to choose, for each dimension, the two or three indicators that carry your
argument, and go back to the section where they are explained.

---
# Part 9. LLM analysis with Gemini

## 9.1 What an LLM adds, and what it does not

The word patterns of notebook 1 find words; they cannot judge whether an explanation is convincing or whether a life
story justifies an exception. The tables of Parts 1 to 8 are precise, but there are many of them, and turning them
into an argument takes reading and judgement. A large language model (LLM) helps with both: it can read a whole form
the way a committee member would, and it can digest a large amount of evidence quickly. We use Gemini for five tasks.

| Section | Task | What the LLM receives |
|---|---|---|
| 9.3 | **Second reader** of the most disputed applicants | the call's rules and the complete form, *not* the two decisions |
| 9.4 | **Procedure reconstruction**, for each allocation separately and blind | the evidence of Parts 3 to 5 for one allocation, labelled only "allocation X" |
| 9.5 | **Check of your declared procedure** | your `OWN_PROCEDURE` and `OWN_DECLARED_PRIORITIES`, and the evidence about your allocation |
| 9.6 | **Devil's advocate** on the four graded dimensions | the scorecard and the tables of Part 8 |
| 9.7 | **Synthesis** | a compact evidence pack from the whole notebook |

Three cautions apply. An LLM can state a plausible reason with more confidence than the evidence supports. It shares
the verbosity bias discussed in notebook 1 (2.15) (Zheng et al., 2023; Saito et al., 2023). And its answer depends on
the prompt and changes from one run to the next. We reduce these risks by asking for quotations and numbers, by fixing
the output format (JSON), and by hiding information that would anchor it (the decisions in 9.3; which allocation is
yours in 9.4). **Treat every LLM answer as a hypothesis from a well-read colleague, and check it against Parts 1
to 8.** Run a call twice: what changes between the two answers is what the evidence does not determine.

## 9.2 How the call to Gemini works

`call_gemini` sends a prompt to the model `GEMINI_MODEL` through Google's `google-genai` library and asks for a JSON
answer. It reads your API key from Colab's *Secrets* (`GEMINI_API_KEY`); the key never appears in the notebook, so
you can share the notebook safely. We leave temperature and the other sampling settings at their defaults, as Google
recommends for the Gemini 3 family. Each answer is saved in your output folder (`llm_answers.json`), with every prompt
(`llm_prompts.txt`), so that the report can quote exactly what the model was asked and what it said.

Without a key (`USE_GEMINI = False`), the notebook still runs. For the two example files, it shows stored answers
from `llm_cache.json`. **These stored answers were written by Claude (Anthropic's model) from the same prompts while
this notebook was being prepared, not by Gemini**; they illustrate the kind of output to expect. For your own files
there is no stored answer: the notebook saves the prompts, which you can also paste into Gemini in your browser.

In [ ]:
CACHE_FILE = "llm_cache.json"
LLM_CACHE = json.load(open(CACHE_FILE, encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}
PAIR_ID = hashlib.sha1(PAIR.round(0).to_csv().encode()).hexdigest()[:12]   # identifies this pair of files
PROMPT_FILE, ANSWER_FILE = os.path.join(OUTPUT_DIR, "llm_prompts.txt"), os.path.join(OUTPUT_DIR, "llm_answers.json")
open(PROMPT_FILE, "w", encoding="utf-8").close()
LLM_ANSWERS, PROMPT_SHOWN = {}, []
print("Pair fingerprint:", PAIR_ID, "| stored example answers available:", any(k.endswith(PAIR_ID) for k in LLM_CACHE))


def gemini_client():
    from google import genai
    try:
        from google.colab import userdata          # Colab Secrets (key icon in the left bar)
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key)


def parse_json(text):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"unparsed_answer": text}


def call_gemini(prompt, task):
    """Return {'source': ..., 'answer': dict}. With USE_GEMINI, call Gemini; otherwise use a stored answer or save the prompt."""
    key = f"{task}::{PAIR_ID}"
    with open(PROMPT_FILE, "a", encoding="utf-8") as fh:
        fh.write(f"\n\n===== {task} =====\n{prompt}\n")
    if USE_GEMINI:
        from google.genai import types
        response = gemini_client().models.generate_content(
            model=GEMINI_MODEL, contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json"))
        result = {"source": f"Gemini ({GEMINI_MODEL})", "answer": parse_json(response.text)}
    elif key in LLM_CACHE:
        result = LLM_CACHE[key]
    else:
        if not PROMPT_SHOWN:
            print("[No stored answer for this pair of files. Set USE_GEMINI = True, or paste the prompts saved in "
                  f"{PROMPT_FILE} into Gemini. The first prompt starts like this:]\n")
            print(prompt[:2500] + "\n[...]\n")
            PROMPT_SHOWN.append(task)
        result = {"source": "none (prompt saved in llm_prompts.txt)", "answer": {}}
    LLM_ANSWERS[task] = result
    json.dump(LLM_ANSWERS, open(ANSWER_FILE, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
    return result


def render(v, indent=0):
    pad = "  " * indent
    if isinstance(v, dict):
        return "\n".join(f"{pad}{k}:\n{render(x, indent + 1)}" for k, x in v.items())
    if isinstance(v, list):
        return "\n".join(f"{pad}- item {i + 1}\n" + render(x, indent + 1) if isinstance(x, (dict, list)) else
                         textwrap.fill(str(x), 110, initial_indent=pad + "- ", subsequent_indent=pad + "  ") for i, x in enumerate(v)) or pad + "- (none)"
    return textwrap.fill(str(v), 110, initial_indent=pad, subsequent_indent=pad)


def show_answer(result, title):
    text = f"=== {title}  [source: {result['source']}] ===\n" + "\n".join(
        f"\n[{k}]\n{render(v, 1)}" for k, v in result["answer"].items())
    print(text + "\n")
    REPORT.append(("text", f"{title} [source: {result['source']}]", text))


CALL_RULES = """EduAll Grant Call 2026 (summary of the rules relevant here)
- Mission: equal opportunity for international students in financial need studying in Switzerland.
- Budget CHF 500,000; maximum award per year: Bachelor's 8,000, Master's 6,500, Doctoral 5,000.
- Awards cover Sept 2027 - Aug 2028 only, pro-rated for students who finish during that year.
- The award normally does not exceed the documented shortfall for the grant period: expenses minus family support
  and expected personal income, taking other scholarships and available savings into account.
- Criteria: financial need 40%, academic motivation and potential 25%, resilience 20%, diversity 10%, values 5%.
- Eligibility: valid permit (B, C, F, S, L, G); pending permits assessed case by case, award conditional on the permit.
  Permit N (asylum procedure ongoing) not eligible. Permit C holders with significant family financial resources not
  eligible; need is assessed on resources actually available. Study permits require full-time study and allow at most
  15 hours of work per week in semesters. GPA normally >= 2.5; lower GPA may be considered if the reasons are explained
  and improvement is shown. Doctoral applicants must document the lack of university or supervisor funding.
- Grants cannot cover: family members' expenses, travel home, debt repayment, childcare (separate programme)."""


def flat(df):
    """Make table labels plain strings, so that the evidence can be written as JSON."""
    df = df.copy()
    df.index = [" / ".join(map(str, i)) if isinstance(i, tuple) else str(i) for i in df.index]
    df.columns = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
    return df


def r(x, d=2):
    """Round numbers inside nested structures, to keep prompts short."""
    if isinstance(x, dict):
        return {str(k): r(v, d) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [r(v, d) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if not np.isfinite(x) else round(float(x), d) if abs(x) < 100 else int(round(x))
    if isinstance(x, (np.integer, np.bool_)):
        return x.item()
    return x

## 9.3 Second reader for the most disputed applicants

We take the five applicants with the largest difference between the two allocations among those funded by one group
only. For each, the prompt contains the call's rules and the complete form, but neither decision.

In [ ]:
SECOND_READER_PROMPT = """You are a member of the EduAll selection committee. Read the application below and assess it
against the rules of the call. Do not invent facts. Quote the sentences of the application that support each point.

{rules}

APPLICATION {sid}:
{form}

Answer in JSON with exactly these keys:
- "eligibility_issues": list of rule-based concerns (empty list if none), each with a short quotation
- "exceptions_or_special_circumstances": list of circumstances that could justify an exception or more support, each with a quotation
- "documented_need_2027_28": your estimate in CHF of the shortfall for the grant year, with the calculation in one sentence
- "recommendation": "fund", "fund conditionally" or "do not fund", with one sentence of justification
- "confidence": "low", "medium" or "high", with a short reason"""

one_only = PAIR[(PAIR["own"] > 0) != (PAIR["other"] > 0)]
DISPUTED_TOP = (one_only["own"] - one_only["other"]).abs().sort_values(ascending=False).head(5).index.tolist()
SECOND_READER = {}
for sid in DISPUTED_TOP:
    result = call_gemini(SECOND_READER_PROMPT.format(rules=CALL_RULES, sid=sid, form=FORMS[sid]), f"second_reader::{sid}")
    SECOND_READER[sid] = result["answer"]
    print(f"{sid}: own CHF {OWN[sid]:,.0f}, other CHF {OTHER[sid]:,.0f}, documented ceiling CHF {CEILING[sid]:,.0f}")
    show_answer(result, f"Second reader for {sid}")

if SECOND_READER and all(SECOND_READER.values()):
    show_table(pd.DataFrame({"own": OWN[DISPUTED_TOP], "other": OTHER[DISPUTED_TOP], "ceiling": CEILING[DISPUTED_TOP],
                             "LLM recommendation": [str(SECOND_READER[s].get("recommendation", ""))[:90] for s in DISPUTED_TOP]}),
               "Second reader versus the two groups")

**How to use it.** For each disputed applicant, compare the LLM's recommendation with the two decisions, then check
the quotations against the form. Where the LLM agrees with the other group, ask whether your group missed a
sentence; where it agrees with you, check that it did not simply reward a long or moving text.

## 9.4 Procedure reconstruction, blind

Part 5 reconstructed each allocation's procedure with code. Here the LLM receives the same evidence for one
allocation at a time, labelled only "allocation X", and writes the procedure as a committee would describe it, with
the evidence for each step and the observation that would refute it. Because the LLM does not know which allocation
is yours, it cannot be more charitable to one than to the other.

In [ ]:
def allocation_evidence(a, lab):
    sur = S_OWN if lab == "own" else S_OTHER
    sel = SEL_OWN if lab == "own" else SEL_OTHER
    scr = SCR_OWN if lab == "own" else SCR_OTHER
    iw = IW_OWN if lab == "own" else IW_OTHER
    return r({
        "shape": profile(a).to_dict(),
        "audit": {k: v for k, v in flat(AUDIT[[lab]]).to_dict()[lab].items()},
        "inferred_screens": scr[["applicants", "funded inside", "funded outside", "verdict"]].to_dict(orient="index"),
        "selection_tree": {"balanced_accuracy": sel["balanced accuracy"], "AUC": sel["AUC"],
                           "rules": sel["rules"].to_dict(orient="records")},
        "award_formula": FORMULA[lab].to_dict(),
        "implied_weights_linear": {"weights": iw["implied weight"].to_dict(), "R2": R2_IW_OWN if lab == "own" else R2_IW_OTHER},
        "criterion_shares_grouped_shap": FAMILIES[f"{lab}: share of all"].to_dict(),
        "top_drivers_shap": drivers(sur, 8).to_dict(orient="index"),
        "surrogate_fidelity": sur["fidelity"],
        "need_reading_spearman": need_reading(a).to_dict(),
        "flagged_groups_funded": {k: f"{v[lab + ': funded']} of {v['applicants']}" for k, v in FLAG_TABLE.to_dict(orient="index").items()},
        "replay_of_code_reconstruction": REPLAY[lab][2],
    })


RECONSTRUCT_PROMPT = """You are an analyst of grant decisions. A committee allocated the EduAll budget to 500 applicants.
You do not know its method. From the evidence below, which was computed from its allocation file, reconstruct the most
plausible procedure. Use only the evidence; cite the numbers you rely on.

{rules}

EVIDENCE ABOUT ALLOCATION X:
{evidence}

Answer in JSON with exactly these keys:
- "procedure": list of 3 to 5 steps, in order, each one sentence, as the committee would describe it
- "evidence_for_each_step": list with, for each step, the numbers that support it
- "interpretation_of_the_call": how this committee read financial need, the exceptions and the grey zones, in three sentences
- "what_would_refute_it": list of two observations that would show the reconstruction is wrong
- "what_the_evidence_cannot_tell": one or two sentences"""

RECONSTRUCTION = {}
for lab, a in [("own", OWN), ("other", OTHER)]:
    RECONSTRUCTION[lab] = call_gemini(RECONSTRUCT_PROMPT.format(rules=CALL_RULES, evidence=json.dumps(allocation_evidence(a, lab), indent=1, default=str)),
                                      f"reconstruction::{lab}")
    show_answer(RECONSTRUCTION[lab], f"Reconstructed procedure of the {lab} allocation (the LLM saw it as 'allocation X')")

**How to use it.** For the other allocation, the reconstruction is a hypothesis to test in your discussion with the
other group: read them the steps and ask which are wrong. For yours, it is a mirror: does an outsider who sees only
your file describe the procedure you followed?

## 9.5 Does your allocation do what your group says it does?

The LLM compares the procedure and priorities you declared in the configuration cell with the evidence about your
allocation. It must separate three things: points where the evidence confirms your account, points where it
contradicts it, and patterns in your allocation that your account does not mention.

In [ ]:
OWN_CHECK_PROMPT = """A student group describes how it allocated the EduAll budget. Compare the group's own account with
the evidence computed from its allocation file. Be precise and fair: confirm what the evidence supports, name what it
contradicts, and point out patterns the account does not mention. Cite numbers from the evidence for every point.

{rules}

THE GROUP'S ACCOUNT OF ITS PROCEDURE:
{procedure}

THE WEIGHTS THE GROUP INTENDED (share of total): {priorities}

EVIDENCE ABOUT THE GROUP'S ALLOCATION:
{evidence}

SELF-CHECK RESULTS: the applicants whose award departs most from the allocation's own rule:
{deviations}

Answer in JSON with exactly these keys:
- "confirmed": list of points of the account that the evidence supports, each with the supporting numbers
- "contradicted": list of points where the evidence differs from the account, each with "declared", "observed" and "possible_reason"
- "not_mentioned": list of patterns visible in the evidence that the account does not mention
- "intended_vs_measured_weights": two sentences comparing the intended weights with the measured ones
- "questions_for_the_group": list of three questions the group should answer in its report"""

if OWN_PROCEDURE.strip() or OWN_DECLARED_PRIORITIES:
    OWN_CHECK = call_gemini(OWN_CHECK_PROMPT.format(
        rules=CALL_RULES, procedure=OWN_PROCEDURE.strip() or "(not provided)",
        priorities=json.dumps(r(DECLARED.to_dict()) if len(DECLARED) else "(not provided)"),
        evidence=json.dumps({**allocation_evidence(OWN, "own"),
                             "measured_criterion_weights_grouped_shap": r(FAMILIES["own: share among the 4 criteria"].dropna().to_dict())},
                            indent=1, default=str),
        deviations=DEV_OWN[["direction", "award", "rule predicts", "flags"]].to_string()), "own_procedure_check")
    show_answer(OWN_CHECK, "Check of your declared procedure")
else:
    OWN_CHECK = {"source": "skipped", "answer": {}}
    print("OWN_PROCEDURE and OWN_DECLARED_PRIORITIES are empty: the check of your declared procedure is skipped.")

**How to use it.** Each *contradicted* point is either a flaw in the procedure (it did not implement your intention)
or a flaw in the account (the procedure was right but you described it imprecisely). Decide which, and correct the
one that is wrong. Reporting such a correction is exactly the kind of critical reflection the grading grid rewards.

## 9.6 Devil's advocate on the four graded dimensions

Groups tend to read the evidence in their own favour. The LLM is asked to do the opposite: for each dimension, to make
the strongest case for the other allocation and against yours, using only the numbers of Part 8, and to say what
evidence would settle the disagreement.

In [ ]:
DEVIL_PROMPT = """Two committees, "own" and "other", allocated the same EduAll budget to the same 500 applicants. The
"own" committee wants to argue that its allocation is better. Your role is the devil's advocate: for each of the four
dimensions below, make the strongest honest case FOR the other allocation and AGAINST the own allocation, using only
the evidence. Do not invent numbers. If the evidence clearly favours the own allocation on a dimension, say so.

{rules}

EVIDENCE (Part 8 of the comparison notebook):
{evidence}

Answer in JSON with exactly these keys, each an object with the keys "case_for_other", "case_against_own",
"what_would_settle_it":
- "mission_alignment"
- "fairness_distributive_justice"
- "diversity_representation"
- "bias_disparate_impact" """

DEVIL_EVIDENCE = r({
    "scorecard": flat(SCORECARD).to_dict(orient="index"),
    "mission": MISSION.to_dict(), "where_each_franc_goes": SPLIT.to_dict(),
    "distributive_justice": JUSTICE.to_dict(),
    "representation_ratio_vs_needy_largest_differences": flat(REPRESENTATION.assign(
        d=(REPRESENTATION["own: ratio vs needy"] - REPRESENTATION["other: ratio vs needy"]).abs()).sort_values("d", ascending=False)
        .head(8)[["with documented need", "own: ratio vs needy", "other: ratio vs needy"]]).to_dict(orient="index"),
    "groups_under_half_proportional": MISSING,
    "diversity_of_recipients": DIVERSITY.to_dict(),
    "disparate_impact_flags": {k: v for k, v in flat(IMPACT[["own: flag", "other: flag"]]).to_dict(orient="index").items()
                               if v["own: flag"] or v["other: flag"]},
    "unexplained_gaps_part7": flat(GAPS).to_dict(),
    "counterfactual_flip": flat(FLIP).to_dict(), "proxy_auc": PROXY.iloc[:, 0].to_dict(),
})
DEVIL = call_gemini(DEVIL_PROMPT.format(rules=CALL_RULES, evidence=json.dumps(DEVIL_EVIDENCE, indent=1, default=str)), "devils_advocate")
show_answer(DEVIL, "Devil's advocate: the case for the other allocation")

**How to use it.** For each dimension, either rebut the devil's advocate with evidence or concede the point. A report
that anticipates the strongest objection to its own allocation is more convincing than one that lists only its
strengths.

## 9.7 Synthesis of the evidence

The prompt gives the LLM a compact **evidence pack**: the main numbers of Parts 1 to 8, not the raw tables, which an
LLM cannot read reliably. It must answer in a fixed structure.

In [ ]:
def evidence_pack():
    return r({
        "audit": flat(AUDIT).to_dict(),
        "shape": pd.DataFrame({"own": profile(OWN), "other": profile(OTHER)}).to_dict(),
        "disagreement": distance(OWN, OTHER).to_dict(),
        "flagged_groups": FLAG_TABLE.to_dict(orient="index"),
        "drivers_of_difference_own_minus_other": drivers(S_DIFF, 6).to_dict(orient="index"),
        "surrogate_fidelity": {"own": S_OWN["fidelity"], "other": S_OTHER["fidelity"], "difference": S_DIFF["fidelity"]},
        "importance_agreement": AGREEMENT.iloc[:, 0].to_dict(),
        "criterion_weights_grouped_shap": flat(FAMILIES).to_dict(),
        "inferred_screens": flat(SCREENS_TABLE).to_dict(orient="index"),
        "award_formula": FORMULA.loc[["formula", "median share of the gap paid", "ρ award vs ceiling"]].to_dict(),
        "implied_weights": flat(IW[["own: implied", "other: implied"]]).to_dict(),
        "replay": {lab: REPLAY[lab][2] for lab in REPLAY},
        "need_reading": pd.DataFrame({"own": need_reading(OWN), "other": need_reading(OTHER)}).to_dict(),
        "criteria_profile": pd.DataFrame({"own": PROF_OWN, "other": PROF_OTHER}).to_dict(),
        "trade_off": {"verdict": VERDICT, "own_ahead_on": AHEAD_OWN, "other_ahead_on": AHEAD_OTHER},
        "unexplained_gaps": flat(GAPS).to_dict(),
        "scorecard": flat(SCORECARD).to_dict(orient="index"),
        "second_reader": {s: SECOND_READER.get(s, {}).get("recommendation", "") for s in DISPUTED_TOP},
    })


SYNTHESIS_PROMPT = """You analyse grant decisions. Two committees, "own" and "other", allocated the same EduAll budget to the
same 500 applicants. You do not know their methods. Explain how and why their allocations differ, using only the
evidence below. Cite numbers from the evidence for every claim.

{rules}

EVIDENCE:
{evidence}

Answer in JSON with exactly these keys:
- "headline": one sentence stating the essential difference
- "own_rationale": the most plausible decision rule behind the own allocation, in two sentences
- "other_rationale": the same for the other allocation
- "interpretation_of_soft_criteria": how the two committees read financial need, the exceptions and the grey zones differently
- "trade_off": what each allocation gains and gives up relative to the other
- "four_dimensions": an object with the keys "mission_alignment", "fairness_distributive_justice", "diversity_representation", "bias_disparate_impact", each one or two sentences
- "errors_to_report": list of hard-rule breaches or unexplained departures from the call (empty list if none)
- "questions_for_the_other_group": list of three questions, most important first
- "caveats": what this evidence cannot tell"""

SYNTHESIS = call_gemini(SYNTHESIS_PROMPT.format(rules=CALL_RULES, evidence=json.dumps(evidence_pack(), indent=1, default=str)), "synthesis")
show_answer(SYNTHESIS, "Synthesis: own versus other")

---
# Part 10. Summary, checklist and saved report

## 10.1 Summary in numbers

The cell below writes a summary from the results above. It contains no judgement: every sentence restates a number
computed in this notebook.

In [ ]:
def top_drivers(sur, k=3):
    return ", ".join(drivers(sur, k).index)


def top_reading(a):
    nr = need_reading(a)[READINGS]
    return f"{nr.idxmax()} (ρ = {nr.max():.2f})"


def screen_list(scr, verdict):
    return ", ".join(scr.index[scr["verdict"] == verdict]) or "none"


hard = [f"{lab}: " + (", ".join(k[1] for k in AUDIT.index
                                if k[0] == "hard rule" and "total" not in k[1] and AUDIT.at[k, lab] not in (0, False)) or "none")
        for lab in ["own", "other"]]
dist = distance(OWN, OTHER)
gap_rows = [f"{lab}, {probe}: {GAPS.at[probe, (lab, 'gap (CHF)')]:+,.0f} CHF (p = {GAPS.at[probe, (lab, 'p-value')]:.3f})"
            for lab in ["own", "other"] for probe in GAPS.index
            if GAPS.at[probe, (lab, "p-value")] < 0.01 and abs(GAPS.at[probe, (lab, "gap (CHF)")]) >= 150]
flag_diff = FLAG_TABLE.assign(d=(FLAG_TABLE["own: funded"] - FLAG_TABLE["other: funded"]).abs()).sort_values("d", ascending=False).head(3)
impact_flags = {lab: [f"{d.split(' (')[0]}: {g} ({v})" for (d, g), v in IMPACT[f"{lab}: flag"].items() if v and "region" not in d]
                for lab in ["own", "other"]}
CLOSED = "share of the pool's documented need closed"
fam = lambda lab: ", ".join(f"{c} {FAMILIES.at[c, lab + ': share among the 4 criteria']:.0%}" for c in CRITERIA4)
SUMMARY = [
    f"1. Hard rules (Part 1). Breaches: {hard[0]}; {hard[1]}.",
    f"2. Distance (Part 2). {dist['share of budget moved']:.0%} of the budget would have to move to turn one allocation into the other; "
    f"{int(dist['funded by both'])} students are funded by both, {int(dist['funded only by own'])} only by own, "
    f"{int(dist['funded only by other'])} only by other (Jaccard {dist['Jaccard overlap']:.2f}). Own funds {int((OWN > 0).sum())} "
    f"students (mean CHF {OWN[OWN > 0].mean():,.0f}); other funds {int((OTHER > 0).sum())} (mean CHF {OTHER[OTHER > 0].mean():,.0f}).",
    "3. Flagged groups (Part 3), largest differences: " + "; ".join(
        f"{name} (own funds {r_['own: funded']}, other {r_['other: funded']} of {r_['applicants']})" for name, r_ in flag_diff.iterrows()) + ".",
    f"4. Drivers (Part 4). Own: {top_drivers(S_OWN)} (fidelity {S_OWN['fidelity']:.2f}). Other: {top_drivers(S_OTHER)} "
    f"(fidelity {S_OTHER['fidelity']:.2f}). Difference: {top_drivers(S_DIFF)}. Importance rankings agree with ρ = "
    f"{AGREEMENT.iloc[0, 0]:.2f} (SHAP) and {AGREEMENT.iloc[1, 0]:.2f} (permutation).",
    f"5. Weight carried by the criteria (4.8). Own: {fam('own')}. Other: {fam('other')}. Call: need 42%, academic 26%, "
    f"resilience 21%, diversity 11%. Distance from the call: own {DIST_TO_CALL['own']:.2f}, other {DIST_TO_CALL['other']:.2f}"
    + (f"; own measured vs declared {tvd(FAMILIES['own: share among the 4 criteria'], DECLARED):.2f}." if len(DECLARED) else "."),
    f"6. Reconstructed procedure (Part 5). Own excludes: {screen_list(SCR_OWN, 'excluded')}; formula: {FORMULA.at['formula', 'own']}; "
    f"replay Jaccard {REPLAY['own'][2]['Jaccard overlap of recipients']:.2f}. Other excludes: {screen_list(SCR_OTHER, 'excluded')}; "
    f"formula: {FORMULA.at['formula', 'other']}; replay Jaccard {REPLAY['other'][2]['Jaccard overlap of recipients']:.2f}.",
    f"7. Reading of need and trade-off (Part 6). Own follows {top_reading(OWN)}; other follows {top_reading(OTHER)}. {VERDICT}: "
    f"own ahead on {', '.join(AHEAD_OWN) or 'nothing'}; other ahead on {', '.join(AHEAD_OTHER) or 'nothing'}.",
    "8. Unexplained gaps (Part 7, p < 0.01 and at least CHF 150): " + ("; ".join(gap_rows) if gap_rows else "none") + ".",
    f"9. Mission (8.1). Share of budget within documented need: own {MISSION.at['share of budget within documented need', 'own']:.0%}, "
    f"other {MISSION.at['share of budget within documented need', 'other']:.0%}; pool's documented need closed: own "
    f"{MISSION.at[CLOSED, 'own']:.0%}, other {MISSION.at[CLOSED, 'other']:.0%}.",
    f"10. Justice (8.2). Concentration index on need: own {JUSTICE.at['need: concentration index', 'own']:+.2f}, other "
    f"{JUSTICE.at['need: concentration index', 'other']:+.2f}; worst-off fifth funded: own "
    f"{JUSTICE.at['priority: share of the worst-off fifth funded', 'own']:.0%}, other {JUSTICE.at['priority: share of the worst-off fifth funded', 'other']:.0%}; "
    f"consistency among neighbours: own {JUSTICE.at['horizontal equity: consistency of decisions among neighbours', 'own']:.2f}, "
    f"other {JUSTICE.at['horizontal equity: consistency of decisions among neighbours', 'other']:.2f}.",
    f"11. Representation (8.3). Groups with less than half their proportional share: own {'; '.join(MISSING['own']) or 'none'}; "
    f"other {'; '.join(MISSING['other']) or 'none'}.",
    f"12. Disparate impact (8.4), groups flagged outside regions: own {', '.join(impact_flags['own']) or 'none'}; "
    f"other {', '.join(impact_flags['other']) or 'none'}.",
]
SUMMARY_TEXT = "\n".join(textwrap.fill(l, 115, subsequent_indent="    ") for l in SUMMARY)
print(SUMMARY_TEXT)
REPORT.insert(0, ("text", "Summary in numbers", SUMMARY_TEXT))

## 10.2 Checklist for the discussion and the report

1. **Facts first.** Report hard-rule breaches (Part 1). They are errors, not differences of opinion.
2. **Then information.** Did both groups use the application forms? Compare the reading of need (6.1), the
   form-based drivers (4.3, 4.7) and the screens (5.1). A group that used only the CSV could not see completion dates,
   earmarked savings, scholarships that end, or family shocks; say so without blame, and show what changed because
   of it.
3. **Then interpretations.** Go through the flagged groups (3.1), the disputed cases (3.2) and the reconstructed
   procedures (5.5, 9.4). For each difference, state the sentence of the call your group applied and how you read it.
   Ask the other group for theirs.
4. **Then your own allocation.** Use the self-check (4.6, 4.8, 9.5): where does your allocation not do what you
   intended? Say whether the flaw is in the procedure or in the account of it, and what you would change.
5. **Then the four graded dimensions.** For each, choose two or three indicators from the scorecard (8.5), interpret
   them against the mission and the call, and answer the devil's advocate (9.6). Name the principle of justice your
   group followed and what it gives up.
6. **Keep the limits in view.** The text variables rest on word patterns: read the evidence before you rely on them.
   A surrogate explains an allocation only up to its fidelity. Implied weights depend on how the criteria were
   scored. A gap in Part 7 or a flag in 8.4 is a question, not proof of bias. An LLM answer is a hypothesis, not a
   finding.

For the written report, a useful structure is: (a) how your group decided, (b) what the comparison revealed about the
other group's rationale, (c) the four dimensions, with evidence, interpretation, judgement and limits, (d) where you now
think your own allocation was weakest, and (e) what you would change.

## 10.3 Saving the report

The cell below writes an HTML report with the summary, every captioned table and figure, the surrogate trees and the
LLM answers, in the order of the notebook. Open it from your Drive in a browser, or download it. The folder also
contains every table as CSV and every figure as PNG, ready for your report.

In [ ]:
def html_report(path):
    css = ("body{font-family:Helvetica,Arial,sans-serif;max-width:1100px;margin:30px auto;color:#1d2a38;line-height:1.45}"
           "h1{color:#0b3c6f}h2{color:#0b3c6f;border-bottom:2px solid #6baed6;padding-bottom:4px;margin-top:36px}"
           "table{border-collapse:collapse;font-size:12px;margin:8px 0}td,th{border:1px solid #d5dde5;padding:3px 7px}"
           "th{background:#eef3f8}pre{background:#f5f8fb;padding:12px;font-size:12px;white-space:pre-wrap}"
           "img{max-width:100%;border:1px solid #e3e8ee}.cap{font-size:12px;color:#555}")
    parts = [f"<html><head><meta charset='utf-8'><title>EduAll comparison</title><style>{css}</style></head><body>",
             f"<h1>EduAll: comparison of two allocations</h1><p>Own allocation: <b>{own_allocation}</b><br>"
             f"Other allocation: <b>{other_allocation}</b><br>Pair fingerprint: {PAIR_ID}</p>",
             "<p class='cap'>Produced by notebook 2. The numbers are evidence; the interpretation and the judgement belong "
             "to the group's report. LLM answers are hypotheses and name their source.</p>"]
    for kind, title, content in REPORT:
        parts.append(f"<h2>{title}</h2>")
        if kind == "table":
            parts.append(content)
        elif kind == "figure":
            img, cap = content
            parts.append(f"<img src='data:image/png;base64,{img}'><p class='cap'>{cap}</p>")
        else:
            parts.append(f"<pre>{content}</pre>")
    parts.append("</body></html>")
    open(path, "w", encoding="utf-8").write("\n".join(parts))


REPORT_FILE = os.path.join(OUTPUT_DIR, "comparison_report.html")
html_report(REPORT_FILE)
open(os.path.join(OUTPUT_DIR, "summary.txt"), "w", encoding="utf-8").write(SUMMARY_TEXT)
PAIR.assign(difference=PAIR["own"] - PAIR["other"]).join(F[["ceiling_text", "shortfall_text"]]).to_csv(os.path.join(OUTPUT_DIR, "pair_by_applicant.csv"))
n_tab = sum(k == "table" for k, _, _ in REPORT); n_fig = sum(k == "figure" for k, _, _ in REPORT)
print(f"Saved in {OUTPUT_DIR}:\n  comparison_report.html ({n_tab} tables, {n_fig} figures)\n  tables/ ({n_tab} CSV files)\n"
      f"  figures/ ({n_fig} PNG files)\n  summary.txt, pair_by_applicant.csv, llm_answers.json, llm_prompts.txt")

## References

* Aas, K., Jullum, M., & Løland, A. (2021). Explaining individual predictions when features are dependent: More
  accurate approximations to Shapley values. *Artificial Intelligence*, 298, 103502.
* Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
* Dawes, R. M. (1979). The robust beauty of improper linear models in decision making. *American Psychologist*, 34(7),
  571–582.
* Equal Employment Opportunity Commission et al. (1978). Uniform Guidelines on Employee Selection Procedures. 29 CFR
  Part 1607.
* Fisher, A., Rudin, C., & Dominici, F. (2019). All models are wrong, but many are useful: Learning a variable's
  importance by studying an entire class of prediction models simultaneously. *Journal of Machine Learning Research*,
  20(177), 1–81.
* Frankfurt, H. (1987). Equality as a moral ideal. *Ethics*, 98(1), 21–43.
* Hooker, G., Mentch, L., & Zhou, S. (2021). Unrestricted permutation forces extrapolation: Variable importance
  requires at least one more model, or there is no free variable importance. *Statistics and Computing*, 31, 82.
* Jost, L. (2006). Entropy and diversity. *Oikos*, 113(2), 363–375.
* Karren, R. J., & Barringer, M. W. (2002). A review and analysis of the policy-capturing methodology in organizational
  research. *Organizational Research Methods*, 5(4), 337–361.
* Kusner, M. J., Loftus, J., Russell, C., & Silva, R. (2017). Counterfactual fairness. *Advances in Neural Information
  Processing Systems*, 30.
* Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural
  Information Processing Systems*, 30.
* Parfit, D. (1997). Equality and priority. *Ratio*, 10(3), 202–221.
* Pielou, E. C. (1966). The measurement of diversity in different types of biological collections. *Journal of
  Theoretical Biology*, 13, 131–144.
* Saito, K., Wachi, A., Wataoka, K., & Akimoto, Y. (2023). Verbosity bias in preference labeling by large language
  models. *arXiv:2310.10076*.
* Wagstaff, A., Paci, P., & van Doorslaer, E. (1991). On the measurement of inequalities in health. *Social Science &
  Medicine*, 33(5), 545–557.
* Zemel, R., Wu, Y., Swersky, K., Pitassi, T., & Dwork, C. (2013). Learning fair representations. *Proceedings of the
  30th International Conference on Machine Learning*, 325–333.
* Zheng, L., Chiang, W.-L., Sheng, Y., et al. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena.
  *Advances in Neural Information Processing Systems*, 36.